In [1]:
#!pip install tabulate

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from scipy import stats
from IPython.display import display, Markdown
import seaborn as sns
import warnings
from scipy.stats import linregress
from functools import reduce
from urllib.parse import urlencode, urlunparse,quote, quote_plus
from IPython.display import FileLink,display, HTML

In [3]:
warnings.filterwarnings('ignore')

class AnalyseurQualite:
    """Classe pour l'analyse de données qualité avec poids et titres"""
    
    def __init__(self, df):
        """
        Initialise l'analyseur avec un DataFrame
        
        Parameters:
        -----------
        df : DataFrame avec colonnes ['A1000M_Poids', 'A1000M_UV_Auto', 'A1000M_Labo', 'campagne']
        """
        self.df = df.copy()
        self.colonnes_analyse = ['A1000M_Poids', 'A1000M_UV_Auto', 'A1000M_Labo']
        
    def statistiques_globales(self):
        """Calcule les statistiques descriptives globales"""
        stats_dict = {}
        
        for col in self.colonnes_analyse:
            data = self.df[col].dropna()
            stats_dict[col] = {
                'Moyenne': data.mean(),
                'Médiane': data.median(),
                'Écart-type': data.std(),
                'CV (%)': (data.std() / data.mean() * 100) if data.mean() != 0 else np.nan,
                'Min': data.min(),
                'Max': data.max(),
                'Q1': data.quantile(0.25),
                'Q3': data.quantile(0.75),
                'IQR': data.quantile(0.75) - data.quantile(0.25),
                'N': len(data)
            }
        
        return pd.DataFrame(stats_dict).T
    
    def statistiques_par_campagne(self):
        """Calcule les statistiques par campagne"""
        resultats = []
        
        for campagne in self.df['campagne'].unique():
            df_camp = self.df[self.df['campagne'] == campagne]
            
            for col in self.colonnes_analyse:
                data = df_camp[col].dropna()
                if len(data) > 0:
                    resultats.append({
                        'Campagne': campagne,
                        'Variable': col,
                        'Moyenne': data.mean(),
                        'Écart-type': data.std(),
                        'CV (%)': (data.std() / data.mean() * 100) if data.mean() != 0 else np.nan,
                        'Min': data.min(),
                        'Max': data.max(),
                        'N': len(data)
                    })
        
        return pd.DataFrame(resultats)
    
    def analyse_ecarts_labo_auto(self):
        """Analyse les écarts entre laboratoire et autocontrôle"""
        df_valid = self.df[['A1000M_UV_Auto', 'A1000M_Labo', 'campagne']].dropna()
        
        df_valid['Ecart_Absolu'] = df_valid['A1000M_Labo'] - df_valid['A1000M_UV_Auto']
        df_valid['Ecart_Relatif (%)'] = (df_valid['Ecart_Absolu'] / df_valid['A1000M_UV_Auto'] * 100)
        
        stats_globales = {
            'Moyenne des écarts': df_valid['Ecart_Absolu'].mean(),
            'Écart-type des écarts': df_valid['Ecart_Absolu'].std(),
            'Biais moyen (%)': df_valid['Ecart_Relatif (%)'].mean(),
            'Corrélation': df_valid['A1000M_UV_Auto'].corr(df_valid['A1000M_Labo']),
            'N paires': len(df_valid)
        }
        
        # Stats par campagne
        stats_campagne = df_valid.groupby('campagne').agg({
            'Ecart_Absolu': ['mean', 'std', 'count'],
            'Ecart_Relatif (%)': ['mean', 'std']
        }).round(4)
        
        return stats_globales, stats_campagne, df_valid
    
    def rapport_markdown(self):
        """Génère un rapport complet en Markdown"""
        rapport = "# 📊 Rapport d'Analyse Qualité\n\n"
        
        # Statistiques globales
        rapport += "## 1. Statistiques Globales\n\n"
        stats_glob = self.statistiques_globales()
        rapport += stats_glob.to_markdown() + "\n\n"
        
        # Statistiques par campagne
        rapport += "## 2. Statistiques par Campagne\n\n"
        stats_camp = self.statistiques_par_campagne()
        for var in self.colonnes_analyse:
            rapport += f"### {var}\n\n"
            df_var = stats_camp[stats_camp['Variable'] == var].drop('Variable', axis=1)
            rapport += df_var.to_markdown(index=False) + "\n\n"
        
        # Analyse écarts Labo vs Auto
        rapport += "## 3. Analyse Laboratoire vs Autocontrôle\n\n"
        stats_glob_ecart, stats_camp_ecart, _ = self.analyse_ecarts_labo_auto()
        
        rapport += "### Statistiques Globales des Écarts\n\n"
        for key, val in stats_glob_ecart.items():
            rapport += f"- **{key}**: {val:.4f}\n"
        rapport += "\n"
        
        rapport += "### Écarts par Campagne\n\n"
        rapport += stats_camp_ecart.to_markdown() + "\n\n"
        
        # Interprétations
        rapport += "## 4. Observations Clés\n\n"
        
        # CV des poids
        cv_poids = stats_glob.loc['A1000M_Poids', 'CV (%)']
        rapport += f"- **Variabilité des poids**: CV = {cv_poids:.2f}%"
        if cv_poids < 5:
            rapport += " (Excellente reproductibilité)\n"
        elif cv_poids < 10:
            rapport += " (Bonne reproductibilité)\n"
        else:
            rapport += " (Variabilité élevée à investiguer)\n"
        
        # Biais Labo vs Auto
        biais = stats_glob_ecart['Biais moyen (%)']
        rapport += f"- **Biais Labo vs Auto**: {biais:.2f}%"
        if abs(biais) < 1:
            rapport += " (Excellent accord)\n"
        elif abs(biais) < 3:
            rapport += " (Accord acceptable)\n"
        else:
            rapport += " (Écart significatif à investiguer)\n"
        
        # Corrélation
        corr = stats_glob_ecart['Corrélation']
        rapport += f"- **Corrélation Labo-Auto**: {corr:.4f}"
        if corr > 0.95:
            rapport += " (Excellente)\n"
        elif corr > 0.90:
            rapport += " (Bonne)\n"
        else:
            rapport += " (À améliorer)\n"
        
        return rapport
    
    def graphique_distribution_poids(self):
        """Graphique de distribution des poids"""
        fig = make_subplots(
            rows=2, cols=2,
            subplot_titles=('Distribution Globale', 'Par Campagne (Violin)',
                          'Évolution Temporelle', 'Boxplot par Campagne'),
            specs=[[{"secondary_y": False}, {"secondary_y": False}],
                   [{"secondary_y": False}, {"secondary_y": False}]]
        )
        
        # Distribution globale
        fig.add_trace(
            go.Histogram(x=self.df['A1000M_Poids'], name='Poids', 
                        nbinsx=30, marker_color='steelblue'),
            row=1, col=1
        )
        
        # Violin plot par campagne
        for campagne in self.df['campagne'].unique():
            data = self.df[self.df['campagne'] == campagne]['A1000M_Poids']
            fig.add_trace(
                go.Violin(y=data, name=f'Camp. {campagne}', box_visible=True),
                row=1, col=2
            )
        
        # Évolution temporelle
        fig.add_trace(
            go.Scatter(x=self.df.index, y=self.df['A1000M_Poids'],
                      mode='lines+markers', name='Poids', marker=dict(size=4)),
            row=2, col=1
        )
        
        # Boxplot par campagne
        for campagne in self.df['campagne'].unique():
            data = self.df[self.df['campagne'] == campagne]['A1000M_Poids']
            fig.add_trace(
                go.Box(y=data, name=f'Camp. {campagne}'),
                row=2, col=2
            )
        
        fig.update_layout(height=800, title_text="Analyse des Poids", showlegend=True)
        fig.update_xaxes(title_text="Poids (g)", row=1, col=1)
        fig.update_xaxes(title_text="Date", row=2, col=1)
        fig.update_yaxes(title_text="Poids (g)")
        
        return fig
    
    def graphique_titres_comparaison(self):
        """Graphique de comparaison Labo vs Auto"""
        df_valid = self.df[['A1000M_UV_Auto', 'A1000M_Labo', 'campagne']].dropna()
        
        fig = make_subplots(
            rows=2, cols=2,
            subplot_titles=('Corrélation Labo vs Auto', 'Bland-Altman Plot',
                          'Distribution des Écarts', 'Écarts par Campagne'),
            specs=[[{"secondary_y": False}, {"secondary_y": False}],
                   [{"secondary_y": False}, {"secondary_y": False}]]
        )
        
        # Scatter Labo vs Auto
        fig.add_trace(
            go.Scatter(x=df_valid['A1000M_UV_Auto'], y=df_valid['A1000M_Labo'],
                      mode='markers', name='Mesures',
                      marker=dict(color=df_valid['campagne'], colorscale='Viridis',
                                 showscale=True, size=6),
                      text=[f'Camp: {c}' for c in df_valid['campagne']],
                      hovertemplate='Auto: %{x:.3f}<br>Labo: %{y:.3f}<br>%{text}'),
            row=1, col=1
        )
        # Ligne y=x
        min_val = min(df_valid['A1000M_UV_Auto'].min(), df_valid['A1000M_Labo'].min())
        max_val = max(df_valid['A1000M_UV_Auto'].max(), df_valid['A1000M_Labo'].max())
        fig.add_trace(
            go.Scatter(x=[min_val, max_val], y=[min_val, max_val],
                      mode='lines', name='y=x', line=dict(dash='dash', color='red')),
            row=1, col=1
        )
        
        # Bland-Altman
        moyenne = (df_valid['A1000M_UV_Auto'] + df_valid['A1000M_Labo']) / 2
        difference = df_valid['A1000M_Labo'] - df_valid['A1000M_UV_Auto']
        mean_diff = difference.mean()
        std_diff = difference.std()
        
        fig.add_trace(
            go.Scatter(x=moyenne, y=difference, mode='markers', name='Écarts',
                      marker=dict(color=df_valid['campagne'], colorscale='Viridis', size=6)),
            row=1, col=2
        )
        fig.add_hline(y=mean_diff, line_dash="dash", line_color="red", 
                     annotation_text=f"Biais: {mean_diff:.4f}", row=1, col=2)
        fig.add_hline(y=mean_diff + 1.96*std_diff, line_dash="dot", line_color="orange",
                     annotation_text=f"+1.96SD", row=1, col=2)
        fig.add_hline(y=mean_diff - 1.96*std_diff, line_dash="dot", line_color="orange",
                     annotation_text=f"-1.96SD", row=1, col=2)
        
        # Distribution des écarts
        fig.add_trace(
            go.Histogram(x=difference, name='Écarts', nbinsx=30, marker_color='coral'),
            row=2, col=1
        )
        
        # Écarts par campagne
        for campagne in df_valid['campagne'].unique():
            ecarts = df_valid[df_valid['campagne'] == campagne]['A1000M_Labo'] - \
                     df_valid[df_valid['campagne'] == campagne]['A1000M_UV_Auto']
            fig.add_trace(
                go.Box(y=ecarts, name=f'Camp. {campagne}'),
                row=2, col=2
            )
        
        fig.update_layout(height=800, title_text="Analyse Laboratoire vs Autocontrôle", 
                         showlegend=True)
        fig.update_xaxes(title_text="UV Auto", row=1, col=1)
        fig.update_yaxes(title_text="UV Labo", row=1, col=1)
        fig.update_xaxes(title_text="Moyenne (Auto+Labo)/2", row=1, col=2)
        fig.update_yaxes(title_text="Différence (Labo-Auto)", row=1, col=2)
        fig.update_xaxes(title_text="Écart (Labo-Auto)", row=2, col=1)
        fig.update_yaxes(title_text="Écart", row=2, col=2)
        
        return fig
    
    def graphique_evolution_temporelle(self):
        """Graphique d'évolution temporelle de toutes les variables"""
        fig = make_subplots(
            rows=3, cols=1,
            subplot_titles=('Poids', 'Titre UV Autocontrôle', 'Titre UV Laboratoire'),
            shared_xaxes=True,
            vertical_spacing=0.08
        )
        
        # Poids
        for campagne in self.df['campagne'].unique():
            df_camp = self.df[self.df['campagne'] == campagne]
            fig.add_trace(
                go.Scatter(x=df_camp.index, y=df_camp['A1000M_Poids'],
                          mode='lines+markers', name=f'Camp. {campagne} (Poids)',
                          marker=dict(size=4)),
                row=1, col=1
            )
        
        # UV Auto
        for campagne in self.df['campagne'].unique():
            df_camp = self.df[self.df['campagne'] == campagne]
            fig.add_trace(
                go.Scatter(x=df_camp.index, y=df_camp['A1000M_UV_Auto'],
                          mode='lines+markers', name=f'Camp. {campagne} (Auto)',
                          marker=dict(size=4)),
                row=2, col=1
            )
        
        # UV Labo
        for campagne in self.df['campagne'].unique():
            df_camp = self.df[self.df['campagne'] == campagne]
            fig.add_trace(
                go.Scatter(x=df_camp.index, y=df_camp['A1000M_Labo'],
                          mode='lines+markers', name=f'Camp. {campagne} (Labo)',
                          marker=dict(size=4)),
                row=3, col=1
        )
        
        fig.update_layout(height=900, title_text="Évolution Temporelle des Variables",
                         showlegend=True)
        fig.update_xaxes(title_text="Date", row=3, col=1)
        fig.update_yaxes(title_text="Poids (g)", row=1, col=1)
        fig.update_yaxes(title_text="Titre UV", row=2, col=1)
        fig.update_yaxes(title_text="Titre UV", row=3, col=1)
        
        return fig
    
    def matrice_correlation(self, figsize=(12, 10)):
        """
        Matrice de corrélation avec Seaborn et analyse détaillée
        
        Parameters:
        -----------
        figsize : tuple, taille de la figure (largeur, hauteur)
        """
        # Données pour corrélation
        df_corr = self.df[self.colonnes_analyse].dropna()
        
        # Calcul de la matrice de corrélation
        corr_matrix = df_corr.corr()
        
        # Créer la figure avec subplots
        fig, axes = plt.subplots(2, 2, figsize=figsize)
        fig.suptitle('Analyse de Corrélation - Poids et Titres UV', 
                     fontsize=16, fontweight='bold', y=0.995)
        
        # 1. Matrice de corrélation (heatmap)
        ax1 = axes[0, 0]
        sns.heatmap(corr_matrix, annot=True, fmt='.4f', cmap='coolwarm', 
                    center=0, vmin=-1, vmax=1, square=True, 
                    linewidths=2, cbar_kws={"shrink": 0.8}, ax=ax1)
        ax1.set_title('Matrice de Corrélation', fontsize=12, fontweight='bold')
        
        # 2. Scatter: Poids vs UV Auto
        ax2 = axes[0, 1]
        scatter = ax2.scatter(df_corr['A1000M_Poids'], df_corr['A1000M_UV_Auto'],
                             c=self.df.loc[df_corr.index, 'campagne'].astype('category').cat.codes,
                             cmap='viridis', alpha=0.6, s=50, edgecolors='black', linewidth=0.5)
        
        # Ligne de régression
        z = np.polyfit(df_corr['A1000M_Poids'], df_corr['A1000M_UV_Auto'], 1)
        p = np.poly1d(z)
        ax2.plot(df_corr['A1000M_Poids'], p(df_corr['A1000M_Poids']), 
                "r--", linewidth=2, label=f'y={z[0]:.4f}x+{z[1]:.4f}')
        
        corr_val = corr_matrix.loc['A1000M_Poids', 'A1000M_UV_Auto']
        ax2.set_xlabel('Poids (g)', fontsize=10)
        ax2.set_ylabel('UV Autocontrôle', fontsize=10)
        ax2.set_title(f'Poids vs UV Auto (r={corr_val:.4f})', 
                     fontsize=12, fontweight='bold')
        ax2.legend()
        ax2.grid(True, alpha=0.3)
        
        # Colorbar pour les campagnes
        cbar = plt.colorbar(scatter, ax=ax2)
        cbar.set_label('Campagne', rotation=270, labelpad=15)
        
        # 3. Scatter: Poids vs UV Labo
        ax3 = axes[1, 0]
        scatter2 = ax3.scatter(df_corr['A1000M_Poids'], df_corr['A1000M_Labo'],
                              c=self.df.loc[df_corr.index, 'campagne'].astype('category').cat.codes,
                              cmap='viridis', alpha=0.6, s=50, edgecolors='black', linewidth=0.5)
        
        # Ligne de régression
        z2 = np.polyfit(df_corr['A1000M_Poids'], df_corr['A1000M_Labo'], 1)
        p2 = np.poly1d(z2)
        ax3.plot(df_corr['A1000M_Poids'], p2(df_corr['A1000M_Poids']), 
                "r--", linewidth=2, label=f'y={z2[0]:.4f}x+{z2[1]:.4f}')
        
        corr_val2 = corr_matrix.loc['A1000M_Poids', 'A1000M_Labo']
        ax3.set_xlabel('Poids (g)', fontsize=10)
        ax3.set_ylabel('UV Laboratoire', fontsize=10)
        ax3.set_title(f'Poids vs UV Labo (r={corr_val2:.4f})', 
                     fontsize=12, fontweight='bold')
        ax3.legend()
        ax3.grid(True, alpha=0.3)
        
        # Colorbar
        cbar2 = plt.colorbar(scatter2, ax=ax3)
        cbar2.set_label('Campagne', rotation=270, labelpad=15)
        
        # 4. Tableau des corrélations par campagne
        ax4 = axes[1, 1]
        ax4.axis('off')
        
        # Calcul des corrélations par campagne
        corr_campagnes = []
        for campagne in sorted(self.df['campagne'].unique()):
            df_camp = self.df[self.df['campagne'] == campagne][self.colonnes_analyse].dropna()
            if len(df_camp) > 2:
                corr_poids_auto = df_camp['A1000M_Poids'].corr(df_camp['A1000M_UV_Auto'])
                corr_poids_labo = df_camp['A1000M_Poids'].corr(df_camp['A1000M_Labo'])
                corr_auto_labo = df_camp['A1000M_UV_Auto'].corr(df_camp['A1000M_Labo'])
                corr_campagnes.append([
                    f'Camp. {campagne}',
                    f'{corr_poids_auto:.4f}',
                    f'{corr_poids_labo:.4f}',
                    f'{corr_auto_labo:.4f}',
                    len(df_camp)
                ])
        
        # Créer le tableau
        table_data = [['Campagne', 'Poids-Auto', 'Poids-Labo', 'Auto-Labo', 'N']] + corr_campagnes
        table = ax4.table(cellText=table_data, cellLoc='center', loc='center',
                         colWidths=[0.25, 0.2, 0.2, 0.2, 0.15])
        table.auto_set_font_size(False)
        table.set_fontsize(9)
        table.scale(1, 2)
        
        # Style de l'en-tête
        for i in range(5):
            table[(0, i)].set_facecolor('#40466e')
            table[(0, i)].set_text_props(weight='bold', color='white')
        
        # Style des lignes alternées
        for i in range(1, len(table_data)):
            for j in range(5):
                if i % 2 == 0:
                    table[(i, j)].set_facecolor('#f0f0f0')
        
        ax4.set_title('Corrélations par Campagne', 
                     fontsize=12, fontweight='bold', pad=20)
        
        plt.tight_layout()
        return fig
    
    def graphique_correlation_plotly(self):
        """Version interactive Plotly de l'analyse de corrélation"""
        df_corr = self.df[self.colonnes_analyse + ['campagne']].dropna()
        
        fig = make_subplots(
            rows=2, cols=2,
            subplot_titles=('Matrice de Corrélation', 'Poids vs UV Auto',
                          'Poids vs UV Labo', 'Corrélations par Campagne'),
            specs=[[{"type": "heatmap"}, {"type": "scatter"}],
                   [{"type": "scatter"}, {"type": "table"}]],
            vertical_spacing=0.12,
            horizontal_spacing=0.1
        )
        
        # 1. Heatmap de corrélation
        corr_matrix = df_corr[self.colonnes_analyse].corr()
        
        fig.add_trace(
            go.Heatmap(
                z=corr_matrix.values,
                x=['Poids', 'UV Auto', 'UV Labo'],
                y=['Poids', 'UV Auto', 'UV Labo'],
                colorscale='RdBu',
                zmid=0,
                zmin=-1, zmax=1,
                text=corr_matrix.values,
                texttemplate='%{text:.4f}',
                textfont={"size": 12},
                colorbar=dict(title="Corrélation", x=0.46)
            ),
            row=1, col=1
        )
        
        # 2. Scatter Poids vs UV Auto
        for campagne in sorted(df_corr['campagne'].unique()):
            df_camp = df_corr[df_corr['campagne'] == campagne]
            fig.add_trace(
                go.Scatter(
                    x=df_camp['A1000M_Poids'],
                    y=df_camp['A1000M_UV_Auto'],
                    mode='markers',
                    name=f'Camp. {campagne}',
                    marker=dict(size=8),
                    text=[f'Poids: {p:.2f}<br>UV: {u:.4f}' 
                          for p, u in zip(df_camp['A1000M_Poids'], df_camp['A1000M_UV_Auto'])]
                ),
                row=1, col=2
            )
        
        # Ligne de régression
        z = np.polyfit(df_corr['A1000M_Poids'], df_corr['A1000M_UV_Auto'], 1)
        p_reg = np.poly1d(z)
        x_reg = np.linspace(df_corr['A1000M_Poids'].min(), 
                           df_corr['A1000M_Poids'].max(), 100)
        fig.add_trace(
            go.Scatter(x=x_reg, y=p_reg(x_reg), mode='lines',
                      name=f'Régression (r={corr_matrix.loc["A1000M_Poids", "A1000M_UV_Auto"]:.4f})',
                      line=dict(color='red', dash='dash', width=2)),
            row=1, col=2
        )
        
        # 3. Scatter Poids vs UV Labo
        for campagne in sorted(df_corr['campagne'].unique()):
            df_camp = df_corr[df_corr['campagne'] == campagne]
            fig.add_trace(
                go.Scatter(
                    x=df_camp['A1000M_Poids'],
                    y=df_camp['A1000M_Labo'],
                    mode='markers',
                    name=f'Camp. {campagne}',
                    marker=dict(size=8),
                    text=[f'Poids: {p:.2f}<br>UV: {u:.4f}' 
                          for p, u in zip(df_camp['A1000M_Poids'], df_camp['A1000M_Labo'])],
                    showlegend=False
                ),
                row=2, col=1
            )
        
        # Ligne de régression
        z2 = np.polyfit(df_corr['A1000M_Poids'], df_corr['A1000M_Labo'], 1)
        p_reg2 = np.poly1d(z2)
        fig.add_trace(
            go.Scatter(x=x_reg, y=p_reg2(x_reg), mode='lines',
                      name=f'Régression (r={corr_matrix.loc["A1000M_Poids", "A1000M_Labo"]:.4f})',
                      line=dict(color='red', dash='dash', width=2),
                      showlegend=False),
            row=2, col=1
        )
        
        # 4. Tableau des corrélations par campagne
        corr_campagnes = []
        campagnes_list = []
        for campagne in sorted(df_corr['campagne'].unique()):
            df_camp = df_corr[df_corr['campagne'] == campagne]
            if len(df_camp) > 2:
                campagnes_list.append(f'Camp. {campagne}')
                corr_poids_auto = df_camp['A1000M_Poids'].corr(df_camp['A1000M_UV_Auto'])
                corr_poids_labo = df_camp['A1000M_Poids'].corr(df_camp['A1000M_Labo'])
                corr_auto_labo = df_camp['A1000M_UV_Auto'].corr(df_camp['A1000M_Labo'])
                corr_campagnes.append([
                    f'{corr_poids_auto:.4f}',
                    f'{corr_poids_labo:.4f}',
                    f'{corr_auto_labo:.4f}',
                    str(len(df_camp))
                ])
        
        if corr_campagnes:
            header = ['Campagne', 'Poids-Auto', 'Poids-Labo', 'Auto-Labo', 'N']
            cells = [campagnes_list] + list(zip(*corr_campagnes))
            
            fig.add_trace(
                go.Table(
                    header=dict(values=header,
                               fill_color='#40466e',
                               font=dict(color='white', size=12),
                               align='center'),
                    cells=dict(values=cells,
                              fill_color=[['#f0f0f0' if i % 2 else 'white' 
                                          for i in range(len(campagnes_list))]],
                              align='center',
                              font=dict(size=11))
                ),
                row=2, col=2
            )
        
        fig.update_xaxes(title_text="Poids (g)", row=1, col=2)
        fig.update_yaxes(title_text="UV Auto", row=1, col=2)
        fig.update_xaxes(title_text="Poids (g)", row=2, col=1)
        fig.update_yaxes(title_text="UV Labo", row=2, col=1)
        
        fig.update_layout(
            height=900,
            title_text="Analyse de Corrélation - Poids et Titres UV",
            showlegend=True
        )
        
        return fig
    
    def analyse_complete(self):
        """Lance une analyse complète avec rapport et graphiques"""
        # Afficher le rapport Markdown
        display(Markdown(self.rapport_markdown()))
        
        # Afficher les graphiques
        print("\n" + "="*80)
        print("GRAPHIQUES INTERACTIFS")
        print("="*80 + "\n")
        
        self.graphique_distribution_poids().show()
        self.graphique_titres_comparaison().show()
        self.graphique_evolution_temporelle().show()
        self.graphique_correlation_plotly().show()
        
        print("\n" + "="*80)
        print("MATRICE DE CORRÉLATION (Seaborn)")
        print("="*80 + "\n")
        
        self.matrice_correlation()
        plt.show()


# EXEMPLE D'UTILISATION
# =====================
# 
# # Charger vos données
# df_M = pd.read_csv('votre_fichier.csv', index_col='timestamp', parse_dates=True)
# 
# # Créer l'analyseur
# analyseur = AnalyseurQualite(df_M)
# 
# # Lancer l'analyse complète (inclut maintenant les graphiques de corrélation)
# analyseur.analyse_complete()
# 
# # OU analyses individuelles :
# 
# # Rapport uniquement
# display(Markdown(analyseur.rapport_markdown()))
# 
# # Statistiques globales
# stats = analyseur.statistiques_globales()
# print(stats)
# 
# # Statistiques par campagne
# stats_camp = analyseur.statistiques_par_campagne()
# print(stats_camp)
# 
# # Analyse des écarts
# stats_glob, stats_camp, df_ecarts = analyseur.analyse_ecarts_labo_auto()
# 
# # Graphiques individuels
# analyseur.graphique_distribution_poids().show()
# analyseur.graphique_titres_comparaison().show()
# analyseur.graphique_evolution_temporelle().show()
# 
# # NOUVEAUX : Graphiques de corrélation
# analyseur.graphique_correlation_plotly().show()  # Version interactive Plotly
# analyseur.matrice_correlation()  # Version Seaborn (matplotlib)
# plt.show()

In [4]:
def read_cred():
    f = open("../../../cred.txt", "r")
    cred = f.read()
    f.close()
    return cred

def get_OI(url,start,end,interval='PT1M',tag='xx',auth='xx',hS='00',hF='23'):
	url_all =url+'data-reference='+tag+'&aggregation=TIME'+'&aggregation-function=MEAN'+"&from="+start+"T"+hS+"%3A00%3A00.000Z&to="+end+"T"+hF+"%3A59%3A59.000Z&aggregation-period="+interval
	#print(url_all,auth)
	d_data = pd.read_json(url_all,storage_options={ 'Authorization': 'basic '+ auth})
	# print(d_data['values'][0])
	arr = np.asarray(np.asarray(d_data['values'])[0])
	return d_data['values'][0]

def get_data(tags,start, end):
    liste = list(range(0))
    for tag in tags:
        urlTag = quote(tag, safe=':/?#[]@!$&\'()*+,;=')
        data = get_OI(urlBase,start,end,'PT20M',urlTag,credentials,'00','23')
        df = pd.DataFrame(data)
        df['timestamp'] = pd.to_datetime(df['timestamp'])
        df = df.set_index('timestamp')
        df = df.rename(columns={'value':tag})
        liste.append(df)
    return liste

def merge_data(liste):
    df = reduce(lambda left,right : pd.merge(left, right,left_index=True,right_index=True,how='outer'),liste)
    return df

def detecter_campagnes(df, seuil_jours=30):
    """
    Détecte les campagnes et ajoute plusieurs colonnes d'information
    """
    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index)
    
    df = df.sort_index()
    
    # Calculer les différences
    diff_jours = df.index.to_series().diff().dt.days
    
    # Détecter les nouvelles campagnes
    nouvelle_campagne = (diff_jours > seuil_jours) | (diff_jours.isna())
    
    # Numéro de campagne
    df['campagne'] = nouvelle_campagne.cumsum()
    
    # Date de début de chaque campagne
    df['debut_campagne'] = df.groupby('campagne').apply(lambda x: x.index.min()).loc[df['campagne']].values
    
    # Nombre de jours depuis le début de la campagne
    df['jour_campagne'] = (df.index - df['debut_campagne']).days
    
    return df


def detecter_campagnes_v2(df, seuil_jours=3):
    df = df.sort_index().copy()

    # Calcul de l'écart de temps entre deux points consécutifs
    deltas = df.index.to_series().diff()

    # Nouvelle campagne si delta > seuil_jours
    nouvelle_campagne = deltas > pd.Timedelta(days=seuil_jours)

    # Numérotation : on cumule les True
    df["campagne"] = nouvelle_campagne.cumsum() + 1

    # Ajout colonne mois-année au format mm-aa
    df["mois_annee"] = df.index.strftime("%y-%m")
    #df["mois_annee_dt"] = pd.to_datetime(df.index.to_period('M').astype(str))
    df["mois_annee_dt"] = df.index.tz_localize(None).to_period('M').to_timestamp()
    #df["mois_annee_dt"] = pd.to_datetime(df["mois_annee"], format="%m-%y")

    return df

def separer_elements(df):
    """
    Sépare le dataframe en deux : un pour l'élément M et un pour l'élément R.
    
    Parameters:
    -----------
    df : pandas.DataFrame
        DataFrame contenant les colonnes timestamp, M_* et R_*
    
    Returns:
    --------
    tuple (df_M, df_R)
        df_M : DataFrame avec timestamp, M_Poids, M_UV_Auto, M_Labo, M_Num, M_Campagne
        df_R : DataFrame avec timestamp, R_Poids, R_UV_Auto, R_Labo, R_Num, R_Campagne
    """
    
    # DataFrame pour M : ne garder que les lignes où M_Poids est non nul
    df_M = df[df['A1000M_Poids'].notna()][['A1000M_Poids', 'A1000M_UV_Auto', 'A1000M_Labo', 'A1000M_Num', 'M_Campagne']].copy()
    df_M = df_M.rename(columns={'M_Campagne':'campagne'})
    
    # DataFrame pour R : ne garder que les lignes où R_Poids est non nul
    df_R = df[df['A1000R_Poids'].notna()][['A1000R_Poids', 'A1000R_UV_Auto', 'A1000R_Labo', 'A1000R_Num', 'R_Campagne']].copy()
    df_R = df_R.rename(columns={'R_Campagne':'campagne'})
        
    return df_M, df_R

def ajouter_element_campagne(df, fill_empty=False):
    """
    Ajoute trois colonnes au dataframe : 'Element' (M ou R), 'M_Campagne' et 'R_Campagne'.
    
    Parameters:
    -----------
    df : pandas.DataFrame
        DataFrame contenant les colonnes M_Poids et R_Poids
    fill_empty : bool, default=False
        Si True, propage l'élément sur les lignes vides (forward fill)
        Si False, les lignes sans données restent à NaN
    
    Returns:
    --------
    pandas.DataFrame
        DataFrame avec les nouvelles colonnes 'Element', 'M_Campagne' et 'R_Campagne'
    """
    import numpy as np
    
    # Créer une copie pour ne pas modifier l'original
    df_result = df.copy()
    
    # Identifier l'élément actif basé uniquement sur M_Poids et R_Poids
    df_result['Element'] = np.where(
        df_result['A1000M_Poids'].notna(),
        'M',
        np.where(
            df_result['A1000R_Poids'].notna(),
            'R',
            np.nan
        )
    )
    
    # Détecter les transitions d'élément sur toutes les lignes (y compris NaN)
    element_change = (df_result['Element'] != df_result['Element'].shift()) & df_result['Element'].notna()
    
    # Initialiser les colonnes de campagne
    df_result['M_Campagne'] = np.nan
    df_result['R_Campagne'] = np.nan
    
    # Compteurs de campagne
    m_counter = 0
    r_counter = 0
    
    # Parcourir le dataframe pour attribuer les numéros de campagne
    for idx in df_result.index:
        if df_result.loc[idx, 'Element'] == 'M':
            if element_change.loc[idx]:
                m_counter += 1
            df_result.loc[idx, 'M_Campagne'] = m_counter
        elif df_result.loc[idx, 'Element'] == 'R':
            if element_change.loc[idx]:
                r_counter += 1
            df_result.loc[idx, 'R_Campagne'] = r_counter
    
    # S'assurer que les compteurs commencent à 1 si des valeurs existent
    if df_result['M_Campagne'].notna().any() and df_result['M_Campagne'].min() == 0:
        df_result.loc[df_result['M_Campagne'].notna(), 'M_Campagne'] += 1
    if df_result['R_Campagne'].notna().any() and df_result['R_Campagne'].min() == 0:
        df_result.loc[df_result['R_Campagne'].notna(), 'R_Campagne'] += 1
    
    return df_result

In [5]:
def reincrementer_campagnes_par_temps(df, jours_seuil=3):
    """
    Réincrémente les numéros de campagne en fonction des écarts de temps entre les mesures.
    Si l'écart entre deux lignes dépasse le seuil, le numéro de campagne est incrémenté.
    
    Parameters:
    -----------
    df : pandas.DataFrame
        DataFrame avec timestamp en index (datetime) et une colonne de campagne
    jours_seuil : int or float, default=3
        Nombre de jours au-delà duquel on incrémente la campagne
    
    Returns:
    --------
    pandas.DataFrame
        DataFrame avec les numéros de campagne mis à jour
    """

    # Créer une copie pour ne pas modifier l'original
    df_result = df.copy()
    
    # Identifier la colonne de campagne (M_Campagne ou R_Campagne)
    #if 'M_Campagne' in df_result.columns:
    col_campagne = 'campagne'
    #elif 'R_Campagne' in df_result.columns:
    #    col_campagne = 'R_Campagne'
    #else:
    #    raise ValueError("Le DataFrame doit contenir une colonne M_Campagne ou R_Campagne")
    
    # Calculer les écarts de temps entre les lignes
    time_diff = df_result.index.to_series().diff()
    
    # Détecter les sauts de campagne (écart > seuil)
    saut_campagne = time_diff > pd.Timedelta(days=jours_seuil)
    
    # Réincrémenter les campagnes
    df_result[col_campagne] = saut_campagne.cumsum() + 1

    # Ajout colonne mois-année au format mm-aa
    df_result["mois_annee"] = df_result.index.strftime("%y-%m")
    df_result["mois_annee_dt"] = df_result.index.tz_localize(None).to_period('M').to_timestamp()
    return df_result

def filtering(df):
    df = df.dropna(how="all")
    df = df[(df['A1000M_Poids']>700) & (df['A1000M_Poids']<1300)]
    df = df[(df['A1000M_UV_Auto']>800_000) & (df['A1000M_UV_Auto']<1_300_000)]
    df = df[(df['A1000M_Labo']>700_000) & (df['A1000M_Labo']<1_300_000)]
#    df = df.dropna(how="all", subset=['A1000M_Poids','A1000R_Poids'])
    df = df.dropna(how="all", subset=['A1000M_Poids'])
    return df

def ajoute_cumul(df):
    """
    Ajoute deux colonnes au dataframe :
    - cumul_UV : (colonne 0 * colonne 1) / 1_000_000
    - cumul_Labo : (colonne 0 * colonne 2) / 1_000_000
    
    Parameters:
    df : DataFrame pandas
    
    Returns:
    DataFrame avec les nouvelles colonnes
    """
    df['cumul_UV'] = (df.iloc[:, 0] * df.iloc[:, 1]) / 1_000_000
    df['cumul_Labo'] = (df.iloc[:, 0] * df.iloc[:, 2]) / 1_000_000
    
    return df

def ajouter_moyennes_glissantes(df, window=10):
    """
    Ajoute deux colonnes de moyennes pondérées glissantes :
    - cumul_UV : moyenne pondérée de la colonne 2 par la colonne 1 sur les 10 dernières valeurs
    - cumul_Labo : moyenne pondérée de la colonne 3 par la colonne 1 sur les 10 dernières valeurs
    
    Parameters:
    df : DataFrame pandas
    window : int, nombre de valeurs pour la fenêtre glissante (défaut: 10)
    
    Returns:
    DataFrame avec les nouvelles colonnes
    """
    # Extraction des colonnes
    poids = df.iloc[:, 0]
    valeurs_uv = df.iloc[:, 1]
    valeurs_labo = df.iloc[:, 2]
    
    # Calcul de la moyenne pondérée glissante pour UV
    numerateur_uv = (poids * valeurs_uv).rolling(window=window).sum()
    denominateur = poids.rolling(window=window).sum()
    df['cumul_UV'] = numerateur_uv / denominateur
    
    # Calcul de la moyenne pondérée glissante pour Labo
    numerateur_labo = (poids * valeurs_labo).rolling(window=window).sum()
    df['cumul_Labo'] = numerateur_labo / denominateur
    
    return df

def resume_par_campagne(df):
    # On suppose que df contient déjà la colonne "campagne"
    # et que les colonnes sont : Poids, UV, labo, teneur

    # UV pondéré = somme(Poids * UV) / somme(Poids)
    df["UV_pondere"] = df["A1000M_Poids"] * df["A1000M_UV_Auto"]
    df["UV_labo"] = df["A1000M_Poids"] * df["A1000M_Labo"]

    grouped = df.groupby(["campagne","mois_annee","mois_annee_dt"]).agg(
        nb_lots=("A1000M_Poids", "count"),
        poids_total=("A1000M_Poids", "sum"),
        uv_pondere=("UV_labo", lambda x: x.sum() / df.loc[x.index, "A1000M_Poids"].sum())
#        uv_pondere=("UV_pondere", lambda x: x.sum() / df.loc[x.index, "CTY_A1000M_Poids container"].sum())
    )

    return grouped.reset_index()



In [6]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from scipy import stats
from IPython.display import display, Markdown
import warnings
warnings.filterwarnings('ignore')

class AnalyseurQualite:
    """Classe pour l'analyse de données qualité avec poids et titres"""
    
    def __init__(self, df):
        """
        Initialise l'analyseur avec un DataFrame
        
        Parameters:
        -----------
        df : DataFrame avec colonnes ['A1000M_Poids', 'A1000M_UV_Auto', 'A1000M_Labo', 'campagne']
        """
        self.df = df.copy()
        self.colonnes_analyse = ['A1000M_Poids', 'A1000M_UV_Auto', 'A1000M_Labo']
        
    def statistiques_globales(self):
        """Calcule les statistiques descriptives globales"""
        stats_dict = {}
        
        for col in self.colonnes_analyse:
            data = self.df[col].dropna()
            stats_dict[col] = {
                'Moyenne': data.mean(),
                'Médiane': data.median(),
                'Écart-type': data.std(),
                'CV (%)': (data.std() / data.mean() * 100) if data.mean() != 0 else np.nan,
                'Min': data.min(),
                'Max': data.max(),
                'Q1': data.quantile(0.25),
                'Q3': data.quantile(0.75),
                'IQR': data.quantile(0.75) - data.quantile(0.25),
                'N': len(data)
            }
        
        return pd.DataFrame(stats_dict).T
    
    def statistiques_par_campagne(self):
        """Calcule les statistiques par campagne"""
        resultats = []
        
        for campagne in self.df['campagne'].unique():
            df_camp = self.df[self.df['campagne'] == campagne]
            
            for col in self.colonnes_analyse:
                data = df_camp[col].dropna()
                if len(data) > 0:
                    resultats.append({
                        'Campagne': campagne,
                        'Variable': col,
                        'Moyenne': data.mean(),
                        'Écart-type': data.std(),
                        'CV (%)': (data.std() / data.mean() * 100) if data.mean() != 0 else np.nan,
                        'Min': data.min(),
                        'Max': data.max(),
                        'N': len(data)
                    })
        
        return pd.DataFrame(resultats)
    
    def analyse_ecarts_labo_auto(self):
        """Analyse les écarts entre laboratoire et autocontrôle"""
        df_valid = self.df[['A1000M_UV_Auto', 'A1000M_Labo', 'campagne']].dropna()
        
        df_valid['Ecart_Absolu'] = df_valid['A1000M_Labo'] - df_valid['A1000M_UV_Auto']
        df_valid['Ecart_Relatif (%)'] = (df_valid['Ecart_Absolu'] / df_valid['A1000M_UV_Auto'] * 100)
        
        stats_globales = {
            'Moyenne des écarts': df_valid['Ecart_Absolu'].mean(),
            'Écart-type des écarts': df_valid['Ecart_Absolu'].std(),
            'Biais moyen (%)': df_valid['Ecart_Relatif (%)'].mean(),
            'Corrélation': df_valid['A1000M_UV_Auto'].corr(df_valid['A1000M_Labo']),
            'N paires': len(df_valid)
        }
        
        # Stats par campagne
        stats_campagne = df_valid.groupby('campagne').agg({
            'Ecart_Absolu': ['mean', 'std', 'count'],
            'Ecart_Relatif (%)': ['mean', 'std']
        }).round(4)
        
        return stats_globales, stats_campagne, df_valid
    
    def rapport_markdown(self):
        """Génère un rapport complet en Markdown"""
        rapport = "# 📊 Rapport d'Analyse Qualité\n\n"
        
        # Statistiques globales
        rapport += "## 1. Statistiques Globales\n\n"
        stats_glob = self.statistiques_globales()
        rapport += stats_glob.to_markdown() + "\n\n"
        
        # Statistiques par campagne
        rapport += "## 2. Statistiques par Campagne\n\n"
        stats_camp = self.statistiques_par_campagne()
        for var in self.colonnes_analyse:
            rapport += f"### {var}\n\n"
            df_var = stats_camp[stats_camp['Variable'] == var].drop('Variable', axis=1)
            rapport += df_var.to_markdown(index=False) + "\n\n"
        
        # Analyse écarts Labo vs Auto
        rapport += "## 3. Analyse Laboratoire vs Autocontrôle\n\n"
        stats_glob_ecart, stats_camp_ecart, _ = self.analyse_ecarts_labo_auto()
        
        rapport += "### Statistiques Globales des Écarts\n\n"
        for key, val in stats_glob_ecart.items():
            rapport += f"- **{key}**: {val:.4f}\n"
        rapport += "\n"
        
        rapport += "### Écarts par Campagne\n\n"
        rapport += stats_camp_ecart.to_markdown() + "\n\n"
        
        # Interprétations
        rapport += "## 4. Observations Clés\n\n"
        
        # CV des poids
        cv_poids = stats_glob.loc['A1000M_Poids', 'CV (%)']
        rapport += f"- **Variabilité des poids**: CV = {cv_poids:.2f}%"
        if cv_poids < 5:
            rapport += " (Excellente reproductibilité)\n"
        elif cv_poids < 10:
            rapport += " (Bonne reproductibilité)\n"
        else:
            rapport += " (Variabilité élevée à investiguer)\n"
        
        # Biais Labo vs Auto
        biais = stats_glob_ecart['Biais moyen (%)']
        rapport += f"- **Biais Labo vs Auto**: {biais:.2f}%"
        if abs(biais) < 1:
            rapport += " (Excellent accord)\n"
        elif abs(biais) < 3:
            rapport += " (Accord acceptable)\n"
        else:
            rapport += " (Écart significatif à investiguer)\n"
        
        # Corrélation
        corr = stats_glob_ecart['Corrélation']
        rapport += f"- **Corrélation Labo-Auto**: {corr:.4f}"
        if corr > 0.95:
            rapport += " (Excellente)\n"
        elif corr > 0.90:
            rapport += " (Bonne)\n"
        else:
            rapport += " (À améliorer)\n"
        
        return rapport
    
    def graphique_distribution_poids(self):
        """Graphique de distribution des poids"""
        fig = make_subplots(
            rows=2, cols=2,
            subplot_titles=('Distribution Globale', 'Par Campagne (Violin)',
                          'Évolution Temporelle', 'Boxplot par Campagne'),
            specs=[[{"secondary_y": False}, {"secondary_y": False}],
                   [{"secondary_y": False}, {"secondary_y": False}]]
        )
        
        # Distribution globale
        fig.add_trace(
            go.Histogram(x=self.df['A1000M_Poids'], name='Poids', 
                        nbinsx=30, marker_color='steelblue'),
            row=1, col=1
        )
        
        # Violin plot par campagne
        for campagne in self.df['campagne'].unique():
            data = self.df[self.df['campagne'] == campagne]['A1000M_Poids']
            fig.add_trace(
                go.Violin(y=data, name=f'Camp. {campagne}', box_visible=True),
                row=1, col=2
            )
        
        # Évolution temporelle
        fig.add_trace(
            go.Scatter(x=self.df.index, y=self.df['A1000M_Poids'],
                      mode='lines+markers', name='Poids', marker=dict(size=4)),
            row=2, col=1
        )
        
        # Boxplot par campagne
        for campagne in self.df['campagne'].unique():
            data = self.df[self.df['campagne'] == campagne]['A1000M_Poids']
            fig.add_trace(
                go.Box(y=data, name=f'Camp. {campagne}'),
                row=2, col=2
            )
        
        fig.update_layout(height=800, title_text="Analyse des Poids", showlegend=True)
        fig.update_xaxes(title_text="Poids (g)", row=1, col=1)
        fig.update_xaxes(title_text="Date", row=2, col=1)
        fig.update_yaxes(title_text="Poids (g)")
        
        return fig
    
    def graphique_titres_comparaison(self):
        """Graphique de comparaison Labo vs Auto"""
        df_valid = self.df[['A1000M_UV_Auto', 'A1000M_Labo', 'campagne']].dropna()
        
        fig = make_subplots(
            rows=2, cols=2,
            subplot_titles=('Corrélation Labo vs Auto', 'Bland-Altman Plot',
                          'Distribution des Écarts', 'Écarts par Campagne'),
            specs=[[{"secondary_y": False}, {"secondary_y": False}],
                   [{"secondary_y": False}, {"secondary_y": False}]]
        )
        
        # Scatter Labo vs Auto
        fig.add_trace(
            go.Scatter(x=df_valid['A1000M_UV_Auto'], y=df_valid['A1000M_Labo'],
                      mode='markers', name='Mesures',
                      marker=dict(color=df_valid['campagne'], colorscale='Viridis',
                                 showscale=True, size=6),
                      text=[f'Camp: {c}' for c in df_valid['campagne']],
                      hovertemplate='Auto: %{x:.3f}<br>Labo: %{y:.3f}<br>%{text}'),
            row=1, col=1
        )
        # Ligne y=x
        min_val = min(df_valid['A1000M_UV_Auto'].min(), df_valid['A1000M_Labo'].min())
        max_val = max(df_valid['A1000M_UV_Auto'].max(), df_valid['A1000M_Labo'].max())
        fig.add_trace(
            go.Scatter(x=[min_val, max_val], y=[min_val, max_val],
                      mode='lines', name='y=x', line=dict(dash='dash', color='red')),
            row=1, col=1
        )
        
        # Bland-Altman
        moyenne = (df_valid['A1000M_UV_Auto'] + df_valid['A1000M_Labo']) / 2
        difference = df_valid['A1000M_Labo'] - df_valid['A1000M_UV_Auto']
        mean_diff = difference.mean()
        std_diff = difference.std()
        
        fig.add_trace(
            go.Scatter(x=moyenne, y=difference, mode='markers', name='Écarts',
                      marker=dict(color=df_valid['campagne'], colorscale='Viridis', size=6)),
            row=1, col=2
        )
        fig.add_hline(y=mean_diff, line_dash="dash", line_color="red", 
                     annotation_text=f"Biais: {mean_diff:.4f}", row=1, col=2)
        fig.add_hline(y=mean_diff + 1.96*std_diff, line_dash="dot", line_color="orange",
                     annotation_text=f"+1.96SD", row=1, col=2)
        fig.add_hline(y=mean_diff - 1.96*std_diff, line_dash="dot", line_color="orange",
                     annotation_text=f"-1.96SD", row=1, col=2)
        
        # Distribution des écarts
        fig.add_trace(
            go.Histogram(x=difference, name='Écarts', nbinsx=30, marker_color='coral'),
            row=2, col=1
        )
        
        # Écarts par campagne
        for campagne in df_valid['campagne'].unique():
            ecarts = df_valid[df_valid['campagne'] == campagne]['A1000M_Labo'] - \
                     df_valid[df_valid['campagne'] == campagne]['A1000M_UV_Auto']
            fig.add_trace(
                go.Box(y=ecarts, name=f'Camp. {campagne}'),
                row=2, col=2
            )
        
        fig.update_layout(height=800, title_text="Analyse Laboratoire vs Autocontrôle", 
                         showlegend=True)
        fig.update_xaxes(title_text="UV Auto", row=1, col=1)
        fig.update_yaxes(title_text="UV Labo", row=1, col=1)
        fig.update_xaxes(title_text="Moyenne (Auto+Labo)/2", row=1, col=2)
        fig.update_yaxes(title_text="Différence (Labo-Auto)", row=1, col=2)
        fig.update_xaxes(title_text="Écart (Labo-Auto)", row=2, col=1)
        fig.update_yaxes(title_text="Écart", row=2, col=2)
        
        return fig
    
    def graphique_evolution_temporelle(self):
        """Graphique d'évolution temporelle de toutes les variables"""
        fig = make_subplots(
            rows=3, cols=1,
            subplot_titles=('Poids', 'Titre UV Autocontrôle', 'Titre UV Laboratoire'),
            shared_xaxes=True,
            vertical_spacing=0.08
        )
        
        # Poids
        for campagne in self.df['campagne'].unique():
            df_camp = self.df[self.df['campagne'] == campagne]
            fig.add_trace(
                go.Scatter(x=df_camp.index, y=df_camp['A1000M_Poids'],
                          mode='lines+markers', name=f'Camp. {campagne} (Poids)',
                          marker=dict(size=4)),
                row=1, col=1
            )
        
        # UV Auto
        for campagne in self.df['campagne'].unique():
            df_camp = self.df[self.df['campagne'] == campagne]
            fig.add_trace(
                go.Scatter(x=df_camp.index, y=df_camp['A1000M_UV_Auto'],
                          mode='lines+markers', name=f'Camp. {campagne} (Auto)',
                          marker=dict(size=4)),
                row=2, col=1
            )
        
        # UV Labo
        for campagne in self.df['campagne'].unique():
            df_camp = self.df[self.df['campagne'] == campagne]
            fig.add_trace(
                go.Scatter(x=df_camp.index, y=df_camp['A1000M_Labo'],
                          mode='lines+markers', name=f'Camp. {campagne} (Labo)',
                          marker=dict(size=4)),
                row=3, col=1
            )
        
        fig.update_layout(height=900, title_text="Évolution Temporelle des Variables",
                         showlegend=True)
        fig.update_xaxes(title_text="Date", row=3, col=1)
        fig.update_yaxes(title_text="Poids (g)", row=1, col=1)
        fig.update_yaxes(title_text="Titre UV", row=2, col=1)
        fig.update_yaxes(title_text="Titre UV", row=3, col=1)
        
        return fig
    
    def analyse_complete(self):
        """Lance une analyse complète avec rapport et graphiques"""
        # Afficher le rapport Markdown
        display(Markdown(self.rapport_markdown()))
        
        # Afficher les graphiques
        print("\n" + "="*80)
        print("GRAPHIQUES INTERACTIFS")
        print("="*80 + "\n")
        
        self.graphique_distribution_poids().show()
        self.graphique_titres_comparaison().show()
        self.graphique_evolution_temporelle().show()


# EXEMPLE D'UTILISATION
# =====================
# 
# # Charger vos données
# df_M = pd.read_csv('votre_fichier.csv', index_col='timestamp', parse_dates=True)
# 
# # Créer l'analyseur
# analyseur = AnalyseurQualite(df_M)
# 
# # Lancer l'analyse complète
# analyseur.analyse_complete()
# 
# # OU analyses individuelles :
# 
# # Rapport uniquement
# display(Markdown(analyseur.rapport_markdown()))
# 
# # Statistiques globales
# stats = analyseur.statistiques_globales()
# print(stats)
# 
# # Statistiques par campagne
# stats_camp = analyseur.statistiques_par_campagne()
# print(stats_camp)
# 
# # Analyse des écarts
# stats_glob, stats_camp, df_ecarts = analyseur.analyse_ecarts_labo_auto()
# 
# # Graphiques individuels
# analyseur.graphique_distribution_poids().show()
# analyseur.graphique_titres_comparaison().show()
# analyseur.graphique_evolution_temporelle().show()

In [7]:
urlBase = 'https://oianalytics-100.optimistik.fr/api/oianalytics/time-values/query?'
credentials = read_cred()
tags = ['CTY_A1000M_Poids container','CTY_A1000M_Teneur arr. Vit. A (UV)','CTY_A1000M_Titre VA AC','CTY_A1000M_Numéro Container','CTY_A1000R_Poids container','CTY_A1000R_Teneur arr. Vit. A (UV)','CTY_A1000R_Titre VA AC','CTY_A1000R_Numéro Container']; #,'CTY_FHA1000M_Teneur_VA']
start = '2010-01-01'
end = '2025-12-01'
df_list = get_data(tags,start,end)
data = merge_data(df_list)
data_tags = ['A1000M_Poids','A1000M_UV_Auto','A1000M_Labo','A1000M_Num','A1000R_Poids','A1000R_UV_Auto','A1000R_Labo','A1000R_Num']
brute = data.copy()
brute.columns = data_tags
brute.describe()

,A1000M_Poids,A1000M_UV_Auto,A1000M_Labo,A1000M_Num,A1000R_Poids,A1000R_UV_Auto,A1000R_Labo,A1000R_Num
count,13817.000000,3.590170e+05,3.292710e+05,14014.000000,1876.000000,3.488200e+05,3.222310e+05,1877.000000
mean,1062.344597,1.033933e+06,1.018700e+06,93.830588,1059.513682,1.034376e+06,1.043619e+06,90.999600
std,90.454573,5.539502e+04,1.103189e+05,93.621282,78.812478,2.740044e+04,7.354138e+04,63.434397
min,5.000000,0.000000e+00,0.000000e+00,1.000000,193.000000,7.600000e+05,0.000000e+00,1.000000
25%,1022.000000,1.019000e+06,1.000000e+06,39.000000,1024.000000,1.023000e+06,1.029093e+06,37.000000
50%,1070.000000,1.036000e+06,1.025000e+06,88.000000,1062.000000,1.036000e+06,1.040900e+06,83.000000
75%,1109.000000,1.048000e+06,1.045000e+06,133.000000,1100.000000,1.051000e+06,1.063600e+06,132.000000
max,2585.428571,1.027070e+07,9.794000e+06,7113.000000,1310.000000,1.125000e+06,1.049810e+07,1001.000000


In [8]:
data_tags = ['A1000M_Poids','A1000M_UV_Auto','A1000M_Labo','A1000M_Num','A1000R_Poids','A1000R_UV_Auto','A1000R_Labo','A1000R_Num']
brute = data.copy()
brute.columns = data_tags
brute.describe()

,A1000M_Poids,A1000M_UV_Auto,A1000M_Labo,A1000M_Num,A1000R_Poids,A1000R_UV_Auto,A1000R_Labo,A1000R_Num
count,13817.000000,3.590170e+05,3.292710e+05,14014.000000,1876.000000,3.488200e+05,3.222310e+05,1877.000000
mean,1062.344597,1.033933e+06,1.018700e+06,93.830588,1059.513682,1.034376e+06,1.043619e+06,90.999600
std,90.454573,5.539502e+04,1.103189e+05,93.621282,78.812478,2.740044e+04,7.354138e+04,63.434397
min,5.000000,0.000000e+00,0.000000e+00,1.000000,193.000000,7.600000e+05,0.000000e+00,1.000000
25%,1022.000000,1.019000e+06,1.000000e+06,39.000000,1024.000000,1.023000e+06,1.029093e+06,37.000000
50%,1070.000000,1.036000e+06,1.025000e+06,88.000000,1062.000000,1.036000e+06,1.040900e+06,83.000000
75%,1109.000000,1.048000e+06,1.045000e+06,133.000000,1100.000000,1.051000e+06,1.063600e+06,132.000000
max,2585.428571,1.027070e+07,9.794000e+06,7113.000000,1310.000000,1.125000e+06,1.049810e+07,1001.000000


In [9]:
df_ = brute.copy()
df_ = filtering(df_)
# Repérer les changement de campagne de M->R et R->M
df_ = ajouter_element_campagne(df_, fill_empty=False)
# Séparation Campagnes A1000M et A1000R:
df_M, df_R = separer_elements(df_)
# Detect changement de campagne vs jours :
df_M= reincrementer_campagnes_par_temps(df_M, jours_seuil=2)
df_R_updated = reincrementer_campagnes_par_temps(df_R, jours_seuil=2)
df_M = ajouter_moyennes_glissantes(df_M,50)
#result = detecter_campagnes_v2(result, seuil_jours=14)
campagnes = resume_par_campagne(df_M)
tags_ = df_M.columns.tolist()[:-3]
df_M_tags =  ['A1000M_Poids','A1000M_UV_Auto','A1000M_Labo','A1000M_Num','cumul_UV','cumul_Labo']
# plot_data(df_M, df_M_tags)

In [10]:
analyseur = AnalyseurQualite(df_M)
analyseur.analyse_complete()

# 📊 Rapport d'Analyse Qualité

## 1. Statistiques Globales

|                |        Moyenne |        Médiane |   Écart-type |   CV (%) |    Min |           Max |             Q1 |            Q3 |     IQR |     N |
|:---------------|---------------:|---------------:|-------------:|---------:|-------:|--------------:|---------------:|--------------:|--------:|------:|
| A1000M_Poids   | 1069.58        | 1074           |      68.2769 |  6.3835  |    703 | 1295          | 1028           | 1113          |    85   | 12189 |
| A1000M_UV_Auto |    1.03383e+06 |    1.0356e+06  |   25907.3    |  2.50597 | 869000 |    1.135e+06  |    1.02e+06    |    1.049e+06  | 29000   | 12189 |
| A1000M_Labo    |    1.03449e+06 |    1.03491e+06 |   25170      |  2.43309 | 756231 |    1.1873e+06 |    1.01976e+06 |    1.0492e+06 | 29437.2 | 12189 |

## 2. Statistiques par Campagne

### A1000M_Poids

|   Campagne |   Moyenne |   Écart-type |    CV (%) |    Min |   Max |   N |
|-----------:|----------:|-------------:|----------:|-------:|------:|----:|
|          1 |  1073.58  |      65.3553 |   6.08763 |  782   |  1225 |  59 |
|          2 |  1026.34  |      26.3105 |   2.56353 |  988   |  1108 |  25 |
|          3 |  1015.34  |      48.8379 |   4.81001 |  888   |  1114 |  80 |
|          4 |  1043.81  |      53.2911 |   5.10544 |  831   |  1198 |  97 |
|          5 |  1071.87  |      55.3238 |   5.16142 |  960   |  1154 |  47 |
|          6 |  1069.33  |      62.7506 |   5.86823 |  896   |  1160 |  62 |
|          7 |  1066.08  |      55.4655 |   5.20273 |  775   |  1182 |  60 |
|          8 |  1069.4   |      54.7133 |   5.11628 |  911   |  1187 |  86 |
|          9 |  1075.78  |      45.1764 |   4.19942 |  900   |  1149 |  94 |
|         10 |  1083.17  |      52.2705 |   4.82571 |  713   |  1173 | 148 |
|         11 |  1054.7   |      50.2877 |   4.76794 |  912   |  1147 |  79 |
|         12 |  1059.8   |      43.989  |   4.1507  |  842   |  1145 |  54 |
|         13 |  1053.81  |      66.3682 |   6.29795 |  796   |  1190 |  49 |
|         14 |  1061.68  |      34.0123 |   3.20361 |  926   |  1128 |  92 |
|         15 |  1028.48  |      56.8207 |   5.52475 |  825   |  1123 |  81 |
|         16 |  1110     |     nan      | nan       | 1110   |  1110 |   1 |
|         17 |  1047.06  |      50.8725 |   4.85862 |  908   |  1176 |  88 |
|         18 |  1042.16  |      57.6579 |   5.53254 |  735   |  1143 |  84 |
|         19 |   984.647 |      57.3353 |   5.82293 |  832   |  1097 |  58 |
|         20 |  1020.58  |      76.7497 |   7.52023 |  759   |  1179 |  52 |
|         21 |  1067.91  |      41.1876 |   3.85684 |  920   |  1144 | 144 |
|         22 |  1069.22  |      51.8745 |   4.8516  |  772   |  1200 | 197 |
|         23 |  1072.75  |      48.9325 |   4.5614  |  940   |  1188 | 141 |
|         24 |  1047.95  |      37.2755 |   3.557   |  906   |  1141 | 117 |
|         25 |  1034     |      50.7684 |   4.90991 |  900   |  1129 |  47 |
|         26 |  1007.9   |      55.9014 |   5.54634 |  840   |  1133 | 102 |
|         27 |  1045.29  |      52.0568 |   4.98012 |  889   |  1149 |  98 |
|         28 |  1023     |     nan      | nan       | 1023   |  1023 |   1 |
|         29 |  1070.54  |      54.2609 |   5.06855 |  900   |  1172 |  69 |
|         30 |  1074.08  |      61.0797 |   5.68668 |  901   |  1254 | 160 |
|         31 |  1109.55  |      55.3119 |   4.98505 |  950   |  1286 |  82 |
|         32 |  1112.81  |      45.4468 |   4.08398 | 1021   |  1266 |  69 |
|         33 |  1074.22  |      43.2676 |   4.02783 |  977   |  1175 |  23 |
|         34 |  1089.04  |      43.3572 |   3.98125 |  968   |  1220 | 128 |
|         35 |  1079.37  |      49.6428 |   4.59923 |  971   |  1196 | 112 |
|         36 |  1076.84  |      46.8034 |   4.34637 |  910   |  1159 |  68 |
|         37 |  1076.82  |      48.7444 |   4.5267  |  940   |  1188 | 116 |
|         38 |  1022.63  |      61.9165 |   6.05466 |  840   |  1190 | 107 |
|         39 |  1047.9   |      70.9777 |   6.77332 |  876   |  1201 | 110 |
|         40 |  1041.3   |      59.6808 |   5.73134 |  930   |  1182 |  87 |
|         41 |  1011.69  |      52.9204 |   5.23087 |  931   |  1105 |  13 |
|         42 |  1038.23  |      59.5235 |   5.73316 |  826   |  1178 | 149 |
|         43 |  1064.17  |      62.214  |   5.84625 |  886   |  1208 | 115 |
|         44 |  1045.45  |      45.9957 |   4.39961 |  955   |  1187 |  82 |
|         45 |  1042.22  |      63.031  |   6.04777 |  915   |  1196 |  50 |
|         46 |  1028.2   |      57.4515 |   5.5876  |  902   |  1131 |  56 |
|         47 |  1038.74  |      58.7046 |   5.6515  |  873   |  1195 | 128 |
|         48 |  1059.3   |      73.7087 |   6.95824 |  714   |  1253 | 144 |
|         49 |  1051.77  |      59.764  |   5.68226 |  884   |  1180 | 120 |
|         50 |  1057.23  |      59.6774 |   5.64471 |  928   |  1218 | 154 |
|         51 |  1041.63  |      76.2576 |   7.32099 |  849   |  1157 |  28 |
|         52 |  1068.26  |      74.4417 |   6.96852 |  865   |  1238 | 140 |
|         53 |  1056.5   |      60.6429 |   5.73996 |  876   |  1183 | 156 |
|         54 |  1032.04  |      55.6602 |   5.39324 |  910   |  1216 | 121 |
|         55 |  1042.03  |      58.667  |   5.63009 |  904   |  1249 | 169 |
|         56 |  1046.58  |      68.2151 |   6.51791 |  730   |  1242 | 150 |
|         57 |  1038.32  |      67.9822 |   6.54732 |  818   |  1216 | 136 |
|         58 |  1073.97  |      45.918  |   4.27553 |  917   |  1168 |  70 |
|         59 |  1068.83  |      59.2175 |   5.54043 |  911   |  1177 |  90 |
|         60 |  1064.48  |      80.2331 |   7.53733 |  900   |  1269 |  76 |
|         61 |  1087.01  |      56.9753 |   5.24145 |  884   |  1240 | 138 |
|         62 |  1094.76  |      60.1302 |   5.49253 |  942   |  1255 | 187 |
|         63 |  1078.35  |      61.9057 |   5.74078 |  891   |  1205 | 154 |
|         64 |  1097.13  |      58.6477 |   5.34554 |  960   |  1247 | 116 |
|         65 |  1090.17  |      55.0126 |   5.04622 |  931   |  1196 |  72 |
|         66 |  1099.39  |      58.1444 |   5.28878 |  950   |  1200 |  92 |
|         67 |  1066.17  |      55.4199 |   5.19805 |  983   |  1149 |   6 |
|         68 |  1073.72  |      53.1177 |   4.94706 |  928   |  1191 |  29 |
|         69 |  1070.53  |      52.9527 |   4.94639 |  885   |  1180 |  98 |
|         70 |  1083.12  |      52.0049 |   4.80142 |  973   |  1180 |  26 |
|         71 |  1057.59  |      51.6053 |   4.87952 |  946   |  1205 |  67 |
|         72 |  1031     |     nan      | nan       | 1031   |  1031 |   1 |
|         73 |  1070.17  |      58.9601 |   5.50941 |  878   |  1253 |  94 |
|         74 |  1111.15  |      58.4265 |   5.2582  |  956   |  1281 | 158 |
|         75 |  1079.47  |      51.3892 |   4.76058 |  915   |  1196 | 198 |
|         76 |  1098.82  |      50.3036 |   4.57796 |  946   |  1247 | 183 |
|         77 |  1085.15  |      47.125  |   4.3427  |  954   |  1233 | 155 |
|         78 |  1106.5   |      58.4901 |   5.28606 |  962   |  1236 | 170 |
|         79 |  1127.9   |      54.7501 |   4.85416 |  975   |  1246 | 166 |
|         80 |  1130.74  |      51.6704 |   4.56962 |  938   |  1249 | 133 |
|         81 |  1097.62  |      46.8001 |   4.26376 |  995   |  1150 |  16 |
|         82 |  1154.04  |      41.7497 |   3.61771 | 1083   |  1240 |  27 |
|         83 |  1128.01  |      56.339  |   4.99454 |  973   |  1271 |  81 |
|         84 |  1111.2   |      55.0465 |   4.95378 |  917   |  1257 | 124 |
|         85 |  1128.17  |      79.1292 |   7.01396 | 1002   |  1224 |  12 |
|         86 |  1115.25  |      57.8424 |   5.18648 |  933   |  1289 |  96 |
|         87 |  1118.63  |      58.6885 |   5.24646 |  759   |  1217 | 106 |
|         88 |  1127.32  |      29.4491 |   2.61232 | 1058   |  1176 |  38 |
|         89 |  1121.28  |      43.9868 |   3.92291 | 1027   |  1224 |  34 |
|         90 |  1132.5   |      41.5482 |   3.66872 | 1057   |  1240 |  38 |
|         91 |  1116.88  |      75.5266 |   6.76228 |  726   |  1266 | 109 |
|         92 |  1096.3   |      69.218  |   6.31377 |  860   |  1258 | 111 |
|         93 |  1103.07  |      54.9968 |   4.98582 |  936   |  1261 | 111 |
|         94 |  1096.55  |      68.0115 |   6.20229 |  905   |  1269 |  83 |
|         95 |  1086.9   |      44.6231 |   4.10554 |  950   |  1202 |  85 |
|         96 |  1076.7   |      41.0827 |   3.81561 | 1022   |  1145 |  10 |
|         97 |  1103.71  |      53.8377 |   4.87786 |  938   |  1237 | 128 |
|         98 |  1042.48  |      86.695  |   8.31623 |  755   |  1264 | 193 |
|         99 |  1015.97  |      68.5954 |   6.75173 |  780   |  1153 |  95 |
|        100 |  1049.51  |      91.0275 |   8.67333 |  880   |  1265 |  49 |
|        101 |  1082.05  |      84.3416 |   7.79463 |  923   |  1295 |  94 |
|        102 |  1061.3   |      58.2646 |   5.48995 |  909   |  1201 | 120 |
|        103 |  1064.55  |      71.0466 |   6.67385 |  703   |  1208 |  87 |
|        104 |  1010.96  |      54.4485 |   5.38581 |  850   |  1202 |  78 |
|        105 |  1040.41  |      58.3131 |   5.60479 |  914   |  1200 |  76 |
|        106 |  1032.53  |      83.5416 |   8.09093 |  837   |  1272 | 179 |
|        107 |  1056.25  |      72.1093 |   6.8269  |  900   |  1233 |  87 |
|        108 |  1059.1   |      72.8176 |   6.87544 |  860   |  1243 | 124 |
|        109 |  1086     |     nan      | nan       | 1086   |  1086 |   1 |
|        110 |  1087.44  |      71.4133 |   6.56712 |  957   |  1188 |  16 |
|        111 |  1094.58  |      76.6495 |   7.00265 |  960   |  1237 |  19 |
|        112 |  1095.53  |      74.4994 |   6.80032 |  875   |  1290 | 192 |
|        113 |  1061.3   |      81.0043 |   7.63256 |  885   |  1290 |  82 |
|        114 |  1046     |     nan      | nan       | 1046   |  1046 |   1 |
|        115 |  1122     |      66.0878 |   5.89018 | 1038   |  1211 |   6 |
|        116 |  1018     |     127.449  |  12.5196  |  703   |  1169 |  16 |
|        117 |  1053.82  |      77.6547 |   7.36891 |  864   |  1250 |  92 |
|        118 |  1003.27  |      72.4646 |   7.22286 |  729   |  1239 | 103 |
|        119 |  1050.11  |      65.4472 |   6.2324  |  902   |  1261 | 192 |
|        120 |  1057.95  |      73.2756 |   6.92617 |  818   |  1268 | 127 |
|        121 |  1093.28  |      71.2515 |   6.51725 |  910   |  1291 | 107 |
|        122 |  1080.5   |      75.6429 |   7.00075 |  889   |  1272 |  98 |
|        123 |  1070.92  |      66.8847 |   6.2455  |  899.5 |  1256 | 140 |
|        124 |  1086.97  |      73.0272 |   6.71843 |  913   |  1240 |  95 |
|        125 |  1054.68  |      91.2731 |   8.65411 |  782   |  1272 |  81 |
|        126 |  1078.76  |      86.6023 |   8.02793 |  841   |  1256 |  63 |
|        127 |  1016.21  |      55.3412 |   5.44583 |  942   |  1129 |  15 |
|        128 |  1093.3   |      63.5279 |   5.81065 |  957   |  1288 | 146 |
|        129 |  1067.51  |      80.3073 |   7.52285 |  886   |  1270 | 208 |
|        130 |  1068.24  |      79.8202 |   7.47212 |  901   |  1288 | 158 |
|        131 |  1065.04  |      77.1006 |   7.23919 |  902   |  1215 |  23 |
|        132 |  1064.34  |      84.1297 |   7.9044  |  859   |  1239 |  53 |

### A1000M_UV_Auto

|   Campagne |          Moyenne |   Écart-type |      CV (%) |              Min |              Max |   N |
|-----------:|-----------------:|-------------:|------------:|-----------------:|-----------------:|----:|
|          1 |      1.04261e+06 |     17758    |   1.70322   |      1.007e+06   |      1.069e+06   |  59 |
|          2 |      1.009e+06   |         0    |   0         |      1.009e+06   |      1.009e+06   |  25 |
|          3 |      1.04635e+06 |     16717.5  |   1.59769   |      1.009e+06   |      1.073e+06   |  80 |
|          4 |      1.03754e+06 |     10754.7  |   1.03656   |      1.016e+06   |      1.058e+06   |  97 |
|          5 |      1.062e+06   |         0    |   0         |      1.062e+06   |      1.062e+06   |  47 |
|          6 |      1.048e+06   |         0    |   0         |      1.048e+06   |      1.048e+06   |  62 |
|          7 |      1.048e+06   |         0    |   0         |      1.048e+06   |      1.048e+06   |  60 |
|          8 |      1.03722e+06 |     23463.8  |   2.26218   |      1.008e+06   |      1.071e+06   |  86 |
|          9 |      1.04692e+06 |     21761.9  |   2.07865   |      1.008e+06   |      1.079e+06   |  94 |
|         10 |      1.017e+06   |     29108.4  |   2.8622    | 976000           |      1.06e+06    | 148 |
|         11 |      1.10909e+06 |     19187.5  |   1.73002   |      1.044e+06   |      1.115e+06   |  79 |
|         12 |      1.044e+06   |         0    |   0         |      1.044e+06   |      1.044e+06   |  54 |
|         13 |      1.044e+06   |         0    |   0         |      1.044e+06   |      1.044e+06   |  49 |
|         14 |      1.044e+06   |         0    |   0         |      1.044e+06   |      1.044e+06   |  92 |
|         15 |      1.03385e+06 |     39682.3  |   3.83832   | 965000           |      1.068e+06   |  81 |
|         16 |      1.061e+06   |       nan    | nan         |      1.061e+06   |      1.061e+06   |   1 |
|         17 |      1.04033e+06 |     13873.1  |   1.33352   |      1.018e+06   |      1.061e+06   |  88 |
|         18 |      1.05209e+06 |     11334    |   1.07729   |      1.048e+06   |      1.096e+06   |  84 |
|         19 |      1.04271e+06 |     16531.6  |   1.58545   |      1.018e+06   |      1.095e+06   |  58 |
|         20 |      1.04061e+06 |      1012.21 |   0.0972711 |      1.038e+06   |      1.041e+06   |  52 |
|         21 |      1.02638e+06 |     15046    |   1.46594   |      1.008e+06   |      1.054e+06   | 144 |
|         22 |      1.03241e+06 |     18906.1  |   1.83126   |      1.01e+06    |      1.064e+06   | 197 |
|         23 |      1.04095e+06 |      5963.21 |   0.572864  |      1.02829e+06 |      1.047e+06   | 141 |
|         24 |      1.03019e+06 |     23560.6  |   2.28702   |      1e+06       |      1.054e+06   | 117 |
|         25 |      1.04015e+06 |     33698.1  |   3.23974   |      1.006e+06   |      1.079e+06   |  47 |
|         26 |      1.03188e+06 |     17059.9  |   1.65328   |      1.006e+06   |      1.074e+06   | 102 |
|         27 |      1.04583e+06 |      9126.06 |   0.872615  |      1.028e+06   |      1.065e+06   |  98 |
|         28 |      1.028e+06   |       nan    | nan         |      1.028e+06   |      1.028e+06   |   1 |
|         29 |      1.04133e+06 |     10133.9  |   0.973167  |      1.028e+06   |      1.053e+06   |  69 |
|         30 |      1.03571e+06 |     13059.4  |   1.2609    |      1.011e+06   |      1.055e+06   | 160 |
|         31 |      1.04513e+06 |     10987.7  |   1.05132   |      1.026e+06   |      1.0563e+06  |  82 |
|         32 |      1.03548e+06 |     19482.9  |   1.88154   | 998000           |      1.0563e+06  |  69 |
|         33 |      1.03504e+06 |     12650.8  |   1.22225   |      1.013e+06   |      1.061e+06   |  23 |
|         34 |      1.04524e+06 |     10873    |   1.04024   |      1.029e+06   |      1.061e+06   | 128 |
|         35 |      1.03321e+06 |     11636.4  |   1.12623   |      1.022e+06   |      1.079e+06   | 112 |
|         36 |      1.04165e+06 |     10502.8  |   1.00828   |      1.027e+06   |      1.083e+06   |  68 |
|         37 |      1.01246e+06 |     37881    |   3.74148   | 896000           |      1.056e+06   | 116 |
|         38 |      1.05646e+06 |     21963.9  |   2.07901   |      1.0239e+06  |      1.092e+06   | 107 |
|         39 |      1.03658e+06 |     17097.5  |   1.64941   |      1.0097e+06  |      1.0742e+06  | 110 |
|         40 |      1.03055e+06 |     17784.1  |   1.72569   |      1.0075e+06  |      1.0607e+06  |  87 |
|         41 |      1.01886e+06 |     13956.5  |   1.36981   |      1.007e+06   |      1.0388e+06  |  13 |
|         42 |      1.05415e+06 |     20983    |   1.99052   |      1.0063e+06  |      1.09e+06    | 149 |
|         43 |      1.03388e+06 |     24357.5  |   2.35593   |      1.013e+06   |      1.118e+06   | 115 |
|         44 |      1.02483e+06 |     16419.1  |   1.60214   |      1.004e+06   |      1.049e+06   |  82 |
|         45 |      1.03737e+06 |     22638.6  |   2.18232   | 984000           |      1.053e+06   |  50 |
|         46 |      1.01782e+06 |     28682.7  |   2.81804   | 984000           |      1.054e+06   |  56 |
|         47 |      1.02612e+06 |     20130.5  |   1.9618    | 986000           |      1.05e+06    | 128 |
|         48 |      1.03675e+06 |     21328.9  |   2.05729   | 942789           |      1.074e+06   | 144 |
|         49 |      1.03469e+06 |     10217.6  |   0.987509  |      1.0152e+06  |      1.061e+06   | 120 |
|         50 |      1.04082e+06 |     20109    |   1.93204   | 985500           |      1.0773e+06  | 154 |
|         51 |      1.03319e+06 |     15761.9  |   1.52555   |      1.0139e+06  |      1.05093e+06 |  28 |
|         52 |      1.04423e+06 |     14613.6  |   1.39946   |      1.0087e+06  |      1.0624e+06  | 140 |
|         53 |      1.03272e+06 |     30360.4  |   2.93986   | 958700           |      1.07587e+06 | 156 |
|         54 |      1.04101e+06 |     18822.6  |   1.80811   | 986000           |      1.07e+06    | 121 |
|         55 |      1.0366e+06  |     43911.4  |   4.23609   | 956000           |      1.135e+06   | 169 |
|         56 |      1.03945e+06 |     28794.6  |   2.77017   | 960000           |      1.081e+06   | 150 |
|         57 |      1.03854e+06 |     18645.1  |   1.79531   | 980500           |      1.079e+06   | 136 |
|         58 |      1.03225e+06 |     14136.9  |   1.36952   | 990000           |      1.047e+06   |  70 |
|         59 |      1.03692e+06 |     28278.9  |   2.7272    | 999000           |      1.083e+06   |  90 |
|         60 |      1.04975e+06 |     34629.1  |   3.29878   | 992000           |      1.092e+06   |  76 |
|         61 |      1.03678e+06 |     24676.6  |   2.38011   |      1.002e+06   |      1.063e+06   | 138 |
|         62 |      1.0301e+06  |     21688.2  |   2.10545   | 974000           |      1.0515e+06  | 187 |
|         63 |      1.0256e+06  |     35172.2  |   3.42941   | 928200           |      1.1016e+06  | 154 |
|         64 |      1.03332e+06 |     24018.2  |   2.32437   | 987100           |      1.0521e+06  | 116 |
|         65 |      1.03735e+06 |     20358.5  |   1.96255   | 987100           |      1.06e+06    |  72 |
|         66 |      1.03921e+06 |     26961    |   2.59437   | 962000           |      1.079e+06   |  92 |
|         67 | 945000           |         0    |   0         | 945000           | 945000           |   6 |
|         68 |      1.03606e+06 |      8245.25 |   0.795824  |      1.006e+06   |      1.04487e+06 |  29 |
|         69 |      1.02664e+06 |     19395.3  |   1.88919   | 992000           |      1.05697e+06 |  98 |
|         70 |      1.03412e+06 |     19578.4  |   1.89325   |      1.004e+06   |      1.051e+06   |  26 |
|         71 | 994928           |     48918.9  |   4.91683   | 869000           |      1.068e+06   |  67 |
|         72 |      1.035e+06   |       nan    | nan         |      1.035e+06   |      1.035e+06   |   1 |
|         73 |      1.03668e+06 |     35419    |   3.41656   | 961000           |      1.067e+06   |  94 |
|         74 |      1.02585e+06 |     20562.7  |   2.00445   | 956000           |      1.067e+06   | 158 |
|         75 |      1.02705e+06 |     13750.8  |   1.33886   |      1.001e+06   |      1.054e+06   | 198 |
|         76 |      1.03584e+06 |     27325.7  |   2.63802   | 985000           |      1.096e+06   | 183 |
|         77 |      1.02598e+06 |     26261.4  |   2.55964   | 972000           |      1.087e+06   | 155 |
|         78 |      1.02409e+06 |     25002.2  |   2.44141   | 984000           |      1.065e+06   | 170 |
|         79 |      1.03116e+06 |     53580.7  |   5.19618   | 983000           |      1.113e+06   | 166 |
|         80 |      1.0176e+06  |     28760.2  |   2.82628   | 961000           |      1.055e+06   | 133 |
|         81 |      1.02069e+06 |     22288.2  |   2.18364   |      1.003e+06   |      1.058e+06   |  16 |
|         82 |      1.07606e+06 |     15745.8  |   1.46329   |      1.055e+06   |      1.095e+06   |  27 |
|         83 |      1.05758e+06 |     19474.7  |   1.84144   |      1.015e+06   |      1.083e+06   |  81 |
|         84 |      1.03739e+06 |     19456.5  |   1.87553   | 996000           |      1.076e+06   | 124 |
|         85 |      1.05767e+06 |      3446.56 |   0.325865  |      1.053e+06   |      1.06e+06    |  12 |
|         86 |      1.03614e+06 |     22116.8  |   2.13454   |      1.002e+06   |      1.09203e+06 |  96 |
|         87 |      1.04165e+06 |     17836    |   1.71228   |      1.011e+06   |      1.072e+06   | 106 |
|         88 |      1.02242e+06 |      3333.95 |   0.326084  |      1.02e+06    |      1.039e+06   |  38 |
|         89 |      1.03524e+06 |      9819.77 |   0.948554  |      1.025e+06   |      1.049e+06   |  34 |
|         90 |      1.02666e+06 |     16838.4  |   1.64011   | 997000           |      1.041e+06   |  38 |
|         91 |      1.02796e+06 |     17488.6  |   1.70128   | 992000           |      1.069e+06   | 109 |
|         92 |      1.0435e+06  |     17469.9  |   1.67416   |      1.011e+06   |      1.07e+06    | 111 |
|         93 |      1.03264e+06 |     21643.5  |   2.09594   | 981000           |      1.07e+06    | 111 |
|         94 |      1.03378e+06 |     11776.9  |   1.13921   |      1.018e+06   |      1.051e+06   |  83 |
|         95 |      1.03247e+06 |     23680.7  |   2.2936    | 980000           |      1.067e+06   |  85 |
|         96 | 970156           |      8957.78 |   0.923334  | 962000           | 980000           |  10 |
|         97 |      1.03279e+06 |     29314.4  |   2.83837   | 962000           |      1.066e+06   | 128 |
|         98 |      1.03191e+06 |     27245.2  |   2.64027   | 945000           |      1.076e+06   | 193 |
|         99 |      1.01856e+06 |     50862.6  |   4.99358   | 904000           |      1.08e+06    |  95 |
|        100 |      1.02433e+06 |     21402.3  |   2.0894    | 987000           |      1.045e+06   |  49 |
|        101 |      1.02912e+06 |     24886.5  |   2.41823   | 959000           |      1.072e+06   |  94 |
|        102 |      1.02549e+06 |     20187.9  |   1.96861   | 999000           |      1.052e+06   | 120 |
|        103 |      1.03267e+06 |     17745.6  |   1.71842   | 982000           |      1.05e+06    |  87 |
|        104 |      1.03241e+06 |     13816.4  |   1.33827   |      1.005e+06   |      1.046e+06   |  78 |
|        105 |      1.04031e+06 |      9049.38 |   0.86987   |      1.021e+06   |      1.058e+06   |  76 |
|        106 |      1.02476e+06 |     20612.1  |   2.0114    | 967000           |      1.057e+06   | 179 |
|        107 |      1.03067e+06 |     11993.9  |   1.1637    | 989000           |      1.045e+06   |  87 |
|        108 |      1.03097e+06 |     28039    |   2.71967   | 959000           |      1.08372e+06 | 124 |
|        109 |      1.02e+06    |       nan    | nan         |      1.02e+06    |      1.02e+06    |   1 |
|        110 |      1.07388e+06 |      5500    |   0.512164  |      1.067e+06   |      1.078e+06   |  16 |
|        111 |      1.02892e+06 |     13819.4  |   1.3431    |      1.01799e+06 |      1.051e+06   |  19 |
|        112 |      1.0309e+06  |     15313.2  |   1.48542   |      1.002e+06   |      1.06e+06    | 192 |
|        113 |      1.02932e+06 |     17265.9  |   1.67741   |      1.001e+06   |      1.06e+06    |  82 |
|        114 |      1.024e+06   |       nan    | nan         |      1.024e+06   |      1.024e+06   |   1 |
|        115 |      1.01933e+06 |      3614.78 |   0.354622  |      1.017e+06   |      1.024e+06   |   6 |
|        116 |      1.04306e+06 |      9426.69 |   0.903751  |      1.034e+06   |      1.058e+06   |  16 |
|        117 |      1.03947e+06 |     21504.2  |   2.06878   |      1.007e+06   |      1.068e+06   |  92 |
|        118 |      1.03865e+06 |     16146.9  |   1.55461   |      1.002e+06   |      1.059e+06   | 103 |
|        119 |      1.02616e+06 |     23420.4  |   2.28234   | 967000           |      1.063e+06   | 192 |
|        120 |      1.02839e+06 |     18697.5  |   1.81813   | 996000           |      1.065e+06   | 127 |
|        121 |      1.02588e+06 |     22379.1  |   2.18146   | 997000           |      1.075e+06   | 107 |
|        122 |      1.03288e+06 |      6111.76 |   0.591718  |      1.015e+06   |      1.043e+06   |  98 |
|        123 |      1.02817e+06 |     22690.3  |   2.20685   | 987000           |      1.074e+06   | 140 |
|        124 |      1.02274e+06 |     15709.1  |   1.53598   |      1.004e+06   |      1.058e+06   |  95 |
|        125 |      1.02035e+06 |     22312.8  |   2.18679   | 943000           |      1.05e+06    |  81 |
|        126 |      1.00985e+06 |     31885.5  |   3.15745   | 955000           |      1.052e+06   |  63 |
|        127 |      1.04247e+06 |     83750.4  |   8.03387   | 910000           |      1.11e+06    |  15 |
|        128 |      1.01318e+06 |     31605.7  |   3.11945   | 948000           |      1.056e+06   | 146 |
|        129 |      1.02426e+06 |     19103.1  |   1.86507   | 983000           |      1.077e+06   | 208 |
|        130 |      1.02791e+06 |     16051.4  |   1.56156   | 999000           |      1.058e+06   | 158 |
|        131 |      1.07649e+06 |     15788.6  |   1.46667   |      1.053e+06   |      1.105e+06   |  23 |
|        132 |      1.03536e+06 |     16967.1  |   1.63876   |      1.002e+06   |      1.061e+06   |  53 |

### A1000M_Labo

|   Campagne |          Moyenne |   Écart-type |     CV (%) |              Min |              Max |   N |
|-----------:|-----------------:|-------------:|-----------:|-----------------:|-----------------:|----:|
|          1 |      1.03833e+06 |     30329.1  |   2.92096  | 928606           |      1.08904e+06 |  59 |
|          2 |      1.04521e+06 |     21029.1  |   2.01195  |      1.001e+06   |      1.072e+06   |  25 |
|          3 |      1.05474e+06 |     18674.9  |   1.77057  |      1.01285e+06 |      1.093e+06   |  80 |
|          4 |      1.03581e+06 |     12888.3  |   1.24428  |      1.00108e+06 |      1.05838e+06 |  97 |
|          5 |      1.045e+06   |     31586.9  |   3.02267  | 950000           |      1.10838e+06 |  47 |
|          6 |      1.05634e+06 |     19377.8  |   1.83443  |      1.03113e+06 |      1.108e+06   |  62 |
|          7 |      1.01436e+06 |     22936.9  |   2.26123  | 957531           |      1.04816e+06 |  60 |
|          8 |      1.03747e+06 |     19758.6  |   1.9045   | 999900           |      1.07869e+06 |  86 |
|          9 |      1.05355e+06 |     24528.7  |   2.32819  | 977000           |      1.0882e+06  |  94 |
|         10 |      1.03367e+06 |     12518.3  |   1.21105  | 972000           |      1.0589e+06  | 148 |
|         11 |      1.03899e+06 |     12473.7  |   1.20056  |      1.01375e+06 |      1.0589e+06  |  79 |
|         12 |      1.04694e+06 |     13761.5  |   1.31445  |      1.021e+06   |      1.0611e+06  |  54 |
|         13 |      1.04925e+06 |     17426.8  |   1.66089  | 999200           |      1.0842e+06  |  49 |
|         14 |      1.04865e+06 |     18357.8  |   1.75062  |      1.0106e+06  |      1.10308e+06 |  92 |
|         15 |      1.04425e+06 |     30900    |   2.95906  | 972000           |      1.08992e+06 |  81 |
|         16 | 969387           |       nan    | nan        | 969387           | 969387           |   1 |
|         17 |      1.03816e+06 |     16825.3  |   1.62068  | 987500           |      1.0738e+06  |  88 |
|         18 |      1.05655e+06 |     20793.8  |   1.96809  |      1.0144e+06  |      1.0958e+06  |  84 |
|         19 |      1.04492e+06 |     18743.9  |   1.79382  |      1.003e+06   |      1.091e+06   |  58 |
|         20 |      1.03645e+06 |     12003.4  |   1.15813  |      1.017e+06   |      1.0565e+06  |  52 |
|         21 |      1.03703e+06 |     12837.5  |   1.23791  |      1.0084e+06  |      1.07027e+06 | 144 |
|         22 |      1.03438e+06 |     17149.7  |   1.65797  |      1e+06       |      1.09171e+06 | 197 |
|         23 |      1.0317e+06  |      8426.46 |   0.816758 |      1e+06       |      1.0459e+06  | 141 |
|         24 |      1.04178e+06 |     21574.7  |   2.07094  | 984532           |      1.07567e+06 | 117 |
|         25 |      1.05338e+06 |     23703.9  |   2.25028  | 998000           |      1.081e+06   |  47 |
|         26 |      1.0355e+06  |     15113.1  |   1.45949  |      1e+06       |      1.084e+06   | 102 |
|         27 |      1.04222e+06 |     12283    |   1.17855  |      1e+06       |      1.0652e+06  |  98 |
|         28 |      1.01727e+06 |       nan    | nan        |      1.01727e+06 |      1.01727e+06 |   1 |
|         29 |      1.03989e+06 |     14528.6  |   1.39713  |      1.0027e+06  |      1.0748e+06  |  69 |
|         30 |      1.03765e+06 |     14397.7  |   1.38754  |      1e+06       |      1.0747e+06  | 160 |
|         31 |      1.04262e+06 |      9720.47 |   0.932316 |      1e+06       |      1.0605e+06  |  82 |
|         32 |      1.03401e+06 |     17897.5  |   1.73089  |      1e+06       |      1.0605e+06  |  69 |
|         33 |      1.06943e+06 |     26262.8  |   2.45578  |      1.013e+06   |      1.10627e+06 |  23 |
|         34 |      1.04045e+06 |     12370.6  |   1.18896  |      1e+06       |      1.0718e+06  | 128 |
|         35 |      1.0385e+06  |     15005.3  |   1.44491  |      1e+06       |      1.06941e+06 | 112 |
|         36 |      1.04251e+06 |     19058.3  |   1.82812  |      1e+06       |      1.0804e+06  |  68 |
|         37 |      1.02849e+06 |     34854.4  |   3.3889   | 896000           |      1.0944e+06  | 116 |
|         38 |      1.04743e+06 |     24646.4  |   2.35305  |      1e+06       |      1.09767e+06 | 107 |
|         39 |      1.03359e+06 |     19468.9  |   1.88361  | 995659           |      1.071e+06   | 110 |
|         40 |      1.03169e+06 |     22312.4  |   2.1627   | 991022           |      1.0723e+06  |  87 |
|         41 |      1.011e+06   |     10366.4  |   1.02535  | 985000           |      1.02827e+06 |  13 |
|         42 |      1.05275e+06 |     21517.5  |   2.04393  |      1.0079e+06  |      1.09949e+06 | 149 |
|         43 |      1.03067e+06 |     20351    |   1.97455  | 981707           |      1.0984e+06  | 115 |
|         44 |      1.03656e+06 |     14241.9  |   1.37396  |      1.00395e+06 |      1.06549e+06 |  82 |
|         45 |      1.0362e+06  |     16708.1  |   1.61244  | 972200           |      1.057e+06   |  50 |
|         46 |      1.03375e+06 |     27705.3  |   2.68008  | 965300           |      1.09061e+06 |  56 |
|         47 |      1.02955e+06 |     18716.3  |   1.81791  | 987900           |      1.05912e+06 | 128 |
|         48 |      1.03561e+06 |     24772.5  |   2.39206  | 943400           |      1.07995e+06 | 144 |
|         49 |      1.03116e+06 |      8800.65 |   0.85347  |      1.0014e+06  |      1.05825e+06 | 120 |
|         50 |      1.02374e+06 |     16356    |   1.59767  |      1.0011e+06  |      1.063e+06   | 154 |
|         51 |      1.02701e+06 |     15078.5  |   1.46819  |      1e+06       |      1.0522e+06  |  28 |
|         52 |      1.03228e+06 |     17069.1  |   1.65353  |      1e+06       |      1.06401e+06 | 140 |
|         53 |      1.02584e+06 |     32063.8  |   3.12563  | 938429           |      1.079e+06   | 156 |
|         54 |      1.05773e+06 |     24818.5  |   2.3464   |      1e+06       |      1.07705e+06 | 121 |
|         55 |      1.02743e+06 |     43382.1  |   4.22238  | 946100           |      1.1474e+06  | 169 |
|         56 |      1.03279e+06 |     29137.7  |   2.82127  | 942600           |      1.09958e+06 | 150 |
|         57 |      1.03256e+06 |     20693.9  |   2.00413  | 911800           |      1.08e+06    | 136 |
|         58 |      1.06375e+06 |     61898.7  |   5.81894  |      1.0081e+06  |      1.1873e+06  |  70 |
|         59 |      1.03323e+06 |     28660    |   2.77382  | 993500           |      1.1835e+06  |  90 |
|         60 |      1.05351e+06 |     24298.5  |   2.30643  | 990500           |      1.09141e+06 |  76 |
|         61 |      1.03363e+06 |     29175.8  |   2.82264  | 990200           |      1.09182e+06 | 138 |
|         62 |      1.0332e+06  |     21684.9  |   2.09881  | 919300           |      1.06319e+06 | 187 |
|         63 |      1.04159e+06 |     32242.5  |   3.0955   | 942200           |      1.11924e+06 | 154 |
|         64 |      1.02772e+06 |     17349.3  |   1.68813  | 987100           |      1.0683e+06  | 116 |
|         65 |      1.05031e+06 |     24447.1  |   2.32762  | 987100           |      1.0708e+06  |  72 |
|         66 |      1.02996e+06 |     17220.9  |   1.672    | 987100           |      1.0697e+06  |  92 |
|         67 |      1e+06       |         0    |   0        |      1e+06       |      1e+06       |   6 |
|         68 |      1.02335e+06 |      8589.33 |   0.839334 |      1.0155e+06  |      1.0417e+06  |  29 |
|         69 |      1.024e+06   |     14468.5  |   1.41294  | 987300           |      1.0516e+06  |  98 |
|         70 |      1.03494e+06 |     20428.6  |   1.97389  |      1e+06       |      1.05136e+06 |  26 |
|         71 |      1.01334e+06 |     29761    |   2.93691  | 958200           |      1.06132e+06 |  67 |
|         72 |      1.0366e+06  |       nan    | nan        |      1.0366e+06  |      1.0366e+06  |   1 |
|         73 |      1.03717e+06 |     22525.8  |   2.17186  | 969400           |      1.07648e+06 |  94 |
|         74 |      1.03436e+06 |     17746.9  |   1.71573  | 996400           |      1.07189e+06 | 158 |
|         75 |      1.03192e+06 |     16944.6  |   1.64204  | 999200           |      1.07538e+06 | 198 |
|         76 |      1.02748e+06 |     28858    |   2.80861  | 962900           |      1.09569e+06 | 183 |
|         77 |      1.03531e+06 |     24929    |   2.40789  | 985760           |      1.11556e+06 | 155 |
|         78 |      1.038e+06   |     24837    |   2.39278  | 992900           |      1.0712e+06  | 170 |
|         79 |      1.0313e+06  |     39645.3  |   3.84422  | 969579           |      1.14367e+06 | 166 |
|         80 |      1.02827e+06 |     18307.5  |   1.78043  | 976176           |      1.0642e+06  | 133 |
|         81 |      1.00531e+06 |     32140.9  |   3.19711  | 969900           |      1.07554e+06 |  16 |
|         82 |      1.05516e+06 |     48393.3  |   4.58633  | 957700           |      1.11078e+06 |  27 |
|         83 |      1.03587e+06 |     22211.5  |   2.14423  | 934381           |      1.0717e+06  |  81 |
|         84 |      1.03403e+06 |     19757    |   1.91069  | 984192           |      1.06963e+06 | 124 |
|         85 |      1.03777e+06 |     13402.4  |   1.29146  |      1.0178e+06  |      1.05924e+06 |  12 |
|         86 |      1.03505e+06 |     25068.4  |   2.42195  | 999691           |      1.1172e+06  |  96 |
|         87 |      1.03365e+06 |     15627.2  |   1.51184  | 978268           |      1.07187e+06 | 106 |
|         88 |      1.03434e+06 |      7580.76 |   0.732911 |      1.02264e+06 |      1.0565e+06  |  38 |
|         89 |      1.03364e+06 |     13470.5  |   1.30321  | 993400           |      1.05081e+06 |  34 |
|         90 |      1.03582e+06 |     16008.7  |   1.54551  | 998764           |      1.0576e+06  |  38 |
|         91 |      1.02982e+06 |     22407.4  |   2.17584  | 974504           |      1.08057e+06 | 109 |
|         92 |      1.03285e+06 |     22898.4  |   2.21701  | 988200           |      1.07899e+06 | 111 |
|         93 |      1.03265e+06 |     45234    |   4.38037  | 823514           |      1.11295e+06 | 111 |
|         94 |      1.03672e+06 |     23397    |   2.25683  | 986700           |      1.0984e+06  |  83 |
|         95 |      1.02509e+06 |     19929.6  |   1.94418  | 981562           |      1.06561e+06 |  85 |
|         96 |      1.00286e+06 |     14752.7  |   1.47106  | 972432           |      1.01748e+06 |  10 |
|         97 |      1.03542e+06 |     20344.4  |   1.96485  | 990945           |      1.06829e+06 | 128 |
|         98 |      1.03257e+06 |     16944.8  |   1.64103  | 994190           |      1.0744e+06  | 193 |
|         99 |      1.03414e+06 |     28199    |   2.7268   | 931418           |      1.0779e+06  |  95 |
|        100 |      1.03039e+06 |     20728.9  |   2.01175  | 984412           |      1.07453e+06 |  49 |
|        101 |      1.02975e+06 |     28127.6  |   2.73149  | 981429           |      1.09721e+06 |  94 |
|        102 |      1.03298e+06 |     22991.4  |   2.22573  |      1e+06       |      1.0786e+06  | 120 |
|        103 |      1.03004e+06 |     20005.9  |   1.94223  | 980617           |      1.0693e+06  |  87 |
|        104 |      1.03189e+06 |     27616.4  |   2.6763   | 952300           |      1.07e+06    |  78 |
|        105 |      1.03754e+06 |     12022.8  |   1.15878  |      1.01229e+06 |      1.0672e+06  |  76 |
|        106 |      1.03291e+06 |     24180    |   2.34095  | 954951           |      1.08251e+06 | 179 |
|        107 |      1.02311e+06 |     14118.7  |   1.37997  | 984761           |      1.0488e+06  |  87 |
|        108 |      1.03898e+06 |     31726.9  |   3.05365  | 970400           |      1.135e+06   | 124 |
|        109 |      1.0109e+06  |       nan    | nan        |      1.0109e+06  |      1.0109e+06  |   1 |
|        110 |      1.06672e+06 |      6555.57 |   0.614555 |      1.0465e+06  |      1.07503e+06 |  16 |
|        111 |      1.01661e+06 |     13377.3  |   1.31588  |      1e+06       |      1.04727e+06 |  19 |
|        112 |      1.02451e+06 |     15136.3  |   1.47741  | 985100           |      1.08203e+06 | 192 |
|        113 |      1.02269e+06 |     19839.8  |   1.93997  | 992100           |      1.07551e+06 |  82 |
|        114 |      1e+06       |       nan    | nan        |      1e+06       |      1e+06       |   1 |
|        115 |      1.02831e+06 |     20399.9  |   1.98383  |      1e+06       |      1.05187e+06 |   6 |
|        116 |      1.03687e+06 |     24266.2  |   2.34034  |      1e+06       |      1.06363e+06 |  16 |
|        117 |      1.04154e+06 |     25748.1  |   2.47212  | 995606           |      1.0866e+06  |  92 |
|        118 |      1.02757e+06 |     15043.1  |   1.46396  | 989709           |      1.06329e+06 | 103 |
|        119 |      1.02111e+06 |     31248.7  |   3.06026  | 811366           |      1.07431e+06 | 192 |
|        120 |      1.03371e+06 |     20683.7  |   2.00093  | 996500           |      1.0934e+06  | 127 |
|        121 |      1.0346e+06  |     23728.6  |   2.2935   | 991400           |      1.1053e+06  | 107 |
|        122 |      1.03357e+06 |     12777.9  |   1.23629  | 995848           |      1.0619e+06  |  98 |
|        123 |      1.02242e+06 |     21931.1  |   2.14501  | 969000           |      1.07363e+06 | 140 |
|        124 |      1.02446e+06 |     20879.9  |   2.03813  | 984191           |      1.06256e+06 |  95 |
|        125 |      1.02734e+06 |     27897.7  |   2.71552  | 943400           |      1.17134e+06 |  81 |
|        126 |      1.01282e+06 |     48098.7  |   4.74897  | 756231           |      1.06549e+06 |  63 |
|        127 |      1.07546e+06 |     33031.7  |   3.07139  | 995614           |      1.12124e+06 |  15 |
|        128 |      1.02009e+06 |     39593.7  |   3.88141  | 816043           |      1.08218e+06 | 146 |
|        129 |      1.03689e+06 |     17098.6  |   1.64903  | 991111           |      1.07666e+06 | 208 |
|        130 |      1.02229e+06 |     17113.1  |   1.674    | 975486           |      1.0605e+06  | 158 |
|        131 |      1.07279e+06 |     11909.2  |   1.11012  |      1.0515e+06  |      1.09964e+06 |  23 |
|        132 |      1.03044e+06 |     28236.5  |   2.74022  | 993709           |      1.08158e+06 |  53 |

## 3. Analyse Laboratoire vs Autocontrôle

### Statistiques Globales des Écarts

- **Moyenne des écarts**: 660.0587
- **Écart-type des écarts**: 25792.2469
- **Biais moyen (%)**: 0.0973
- **Corrélation**: 0.4903
- **N paires**: 12189.0000

### Écarts par Campagne

|   campagne |   ('Ecart_Absolu', 'mean') |   ('Ecart_Absolu', 'std') |   ('Ecart_Absolu', 'count') |   ('Ecart_Relatif (%)', 'mean') |   ('Ecart_Relatif (%)', 'std') |
|-----------:|---------------------------:|--------------------------:|----------------------------:|--------------------------------:|-------------------------------:|
|          1 |                 -4282.93   |                  23956.9  |                          59 |                         -0.412  |                         2.2963 |
|          2 |                 36212.4    |                  21029.1  |                          25 |                          3.5889 |                         2.0842 |
|          3 |                  8389.79   |                  19616.7  |                          80 |                          0.8165 |                         1.8989 |
|          4 |                 -1725.24   |                  11803.5  |                          97 |                         -0.1623 |                         1.1313 |
|          5 |                -16999.2    |                  31586.9  |                          47 |                         -1.6007 |                         2.9743 |
|          6 |                  8338.21   |                  19377.8  |                          62 |                          0.7956 |                         1.849  |
|          7 |                -33644.6    |                  22936.9  |                          60 |                         -3.2104 |                         2.1886 |
|          8 |                   250.433  |                  13463.1  |                          86 |                          0.0397 |                         1.2919 |
|          9 |                  6626.57   |                  16180.2  |                          94 |                          0.6394 |                         1.5683 |
|         10 |                 16673.1    |                  30469    |                         148 |                          1.7196 |                         3.075  |
|         11 |                -70102.3    |                  23255.5  |                          79 |                         -6.2908 |                         2.0763 |
|         12 |                  2935.19   |                  13761.5  |                          54 |                          0.2811 |                         1.3182 |
|         13 |                  5245.1    |                  17426.8  |                          49 |                          0.5024 |                         1.6692 |
|         14 |                  4649.22   |                  18357.8  |                          92 |                          0.4453 |                         1.7584 |
|         15 |                 10405.6    |                  36000.1  |                          81 |                          1.0986 |                         3.5625 |
|         16 |                -91613.1    |                    nan    |                           1 |                         -8.6346 |                       nan      |
|         17 |                 -2171.3    |                  13405.5  |                          88 |                         -0.2047 |                         1.2792 |
|         18 |                  4456.63   |                  17898.1  |                          84 |                          0.4244 |                         1.7069 |
|         19 |                  2208.95   |                  11999.5  |                          58 |                          0.2148 |                         1.1646 |
|         20 |                 -4158.88   |                  12157.5  |                          52 |                         -0.3994 |                         1.1679 |
|         21 |                 10646.5    |                  18488.7  |                         144 |                          1.0564 |                         1.8022 |
|         22 |                  1969.34   |                  23448.4  |                         197 |                          0.2194 |                         2.2829 |
|         23 |                 -9251.28   |                   9017.78 |                         141 |                         -0.8867 |                         0.865  |
|         24 |                 11591.7    |                  32060.7  |                         117 |                          1.1788 |                         3.1693 |
|         25 |                 13227.9    |                  33032.9  |                          47 |                          1.3488 |                         3.2499 |
|         26 |                  3622.48   |                  16305.5  |                         102 |                          0.3664 |                         1.5744 |
|         27 |                 -3611.23   |                  13543.5  |                          98 |                         -0.34   |                         1.3061 |
|         28 |                -10733.3    |                    nan    |                           1 |                         -1.0441 |                       nan      |
|         29 |                 -1445.64   |                  14988.3  |                          69 |                         -0.1335 |                         1.4478 |
|         30 |                  1931.33   |                  17459    |                         160 |                          0.199  |                         1.6868 |
|         31 |                 -2516.29   |                  14131.1  |                          82 |                         -0.2305 |                         1.3534 |
|         32 |                 -1470.24   |                   7619.91 |                          69 |                         -0.1367 |                         0.732  |
|         33 |                 34384.9    |                  21558.7  |                          23 |                          3.3195 |                         2.0877 |
|         34 |                 -4782.21   |                  18309    |                         128 |                         -0.444  |                         1.7403 |
|         35 |                  5283.54   |                  12284.1  |                         112 |                          0.5142 |                         1.195  |
|         36 |                   854.412  |                  18392.2  |                          68 |                          0.0859 |                         1.7644 |
|         37 |                 16027.6    |                  25627.6  |                         116 |                          1.6316 |                         2.6737 |
|         38 |                 -9034.1    |                  14468.7  |                         107 |                         -0.8517 |                         1.3577 |
|         39 |                 -2983.87   |                  13293.8  |                         110 |                         -0.2839 |                         1.2803 |
|         40 |                  1142.1    |                  14658.1  |                          87 |                          0.1124 |                         1.4277 |
|         41 |                 -7856.63   |                  15769    |                          13 |                         -0.7565 |                         1.5188 |
|         42 |                 -1396.11   |                  15418.5  |                         149 |                         -0.1227 |                         1.4644 |
|         43 |                 -3215.45   |                  23989    |                         115 |                         -0.2782 |                         2.2374 |
|         44 |                 11732.6    |                  20325.1  |                          82 |                          1.1675 |                         1.9899 |
|         45 |                 -1167.71   |                  19073.1  |                          50 |                         -0.0842 |                         1.9009 |
|         46 |                 15927.2    |                  31083    |                          56 |                          1.6146 |                         3.0781 |
|         47 |                  3431.08   |                  21235.7  |                         128 |                          0.359  |                         2.1044 |
|         48 |                 -1132.49   |                  19900    |                         144 |                         -0.0978 |                         1.9586 |
|         49 |                 -3525.49   |                  11990.7  |                         120 |                         -0.3329 |                         1.1469 |
|         50 |                -17081.2    |                  20953.5  |                         154 |                         -1.6153 |                         1.988  |
|         51 |                 -6178.97   |                  14688.4  |                          28 |                         -0.5875 |                         1.4167 |
|         52 |                -11947.4    |                  21837.6  |                         140 |                         -1.126  |                         2.0863 |
|         53 |                 -6880.39   |                  33516    |                         156 |                         -0.6177 |                         3.2656 |
|         54 |                 16716.5    |                  33308.1  |                         121 |                          1.6458 |                         3.243  |
|         55 |                 -9171.36   |                  32932.3  |                         169 |                         -0.8345 |                         3.1317 |
|         56 |                 -6669.22   |                  17763.9  |                         150 |                         -0.628  |                         1.7234 |
|         57 |                 -5976.24   |                  16961    |                         136 |                         -0.5662 |                         1.631  |
|         58 |                 31498.4    |                  62024.4  |                          70 |                          3.0622 |                         5.9946 |
|         59 |                 -3692.09   |                  19952    |                          90 |                         -0.3391 |                         1.9121 |
|         60 |                  3755.98   |                  28160.2  |                          76 |                          0.4223 |                         2.7357 |
|         61 |                 -3148.48   |                  34200.2  |                         138 |                         -0.2612 |                         3.2732 |
|         62 |                  3101.14   |                  26985.1  |                         187 |                          0.3362 |                         2.6507 |
|         63 |                 15988.9    |                  26334.8  |                         154 |                          1.605  |                         2.6669 |
|         64 |                 -5598.66   |                  21434.6  |                         116 |                         -0.5075 |                         2.071  |
|         65 |                 12957.8    |                  18799.5  |                          72 |                          1.2576 |                         1.8283 |
|         66 |                 -9255.9    |                  27093.7  |                          92 |                         -0.837  |                         2.6344 |
|         67 |                 55000      |                      0    |                           6 |                          5.8201 |                         0      |
|         68 |                -12713.4    |                   9131.2  |                          29 |                         -1.2236 |                         0.881  |
|         69 |                 -2645.62   |                  16299.6  |                          98 |                         -0.2376 |                         1.5832 |
|         70 |                   825.096  |                   3912.5  |                          26 |                          0.0789 |                         0.3843 |
|         71 |                 18414.9    |                  35459.6  |                          67 |                          2.0005 |                         3.7788 |
|         72 |                  1600      |                    nan    |                           1 |                          0.1546 |                       nan      |
|         73 |                   481.557  |                  30897.9  |                          94 |                          0.1272 |                         3.0421 |
|         74 |                  8511.48   |                  22894.9  |                         158 |                          0.8611 |                         2.3152 |
|         75 |                  4865.53   |                  13393.1  |                         198 |                          0.4777 |                         1.3096 |
|         76 |                 -8357.27   |                  22948.1  |                         183 |                         -0.7872 |                         2.1971 |
|         77 |                  9325.44   |                  17923.6  |                         155 |                          0.9279 |                         1.7529 |
|         78 |                 13909.5    |                  16014.5  |                         170 |                          1.3715 |                         1.565  |
|         79 |                   139.782  |                  55029.1  |                         166 |                          0.2092 |                         5.247  |
|         80 |                 10668.1    |                  28109.4  |                         133 |                          1.1117 |                         2.8165 |
|         81 |                -15375.4    |                  29933.1  |                          16 |                         -1.4909 |                         2.9181 |
|         82 |                -20892.2    |                  53086.2  |                          27 |                         -1.9112 |                         4.9479 |
|         83 |                -21705.4    |                  22283.6  |                          81 |                         -2.0359 |                         2.0995 |
|         84 |                 -3361.95   |                  13118.5  |                         124 |                         -0.3169 |                         1.2527 |
|         85 |                -19894.7    |                  15812.2  |                          12 |                         -1.8776 |                         1.4926 |
|         86 |                 -1088.1    |                  22406.5  |                          96 |                         -0.0886 |                         2.1457 |
|         87 |                 -7998.87   |                  17210.8  |                         106 |                         -0.7513 |                         1.6373 |
|         88 |                 11914.6    |                   7872.47 |                          38 |                          1.1661 |                         0.771  |
|         89 |                 -1596.26   |                  17035.2  |                          34 |                         -0.1449 |                         1.6449 |
|         90 |                  9154.79   |                  18624.2  |                          38 |                          0.9095 |                         1.8356 |
|         91 |                  1860.55   |                  18226.8  |                         109 |                          0.1875 |                         1.7923 |
|         92 |                -10650      |                  21312.4  |                         111 |                         -1.0101 |                         2.0402 |
|         93 |                    13.0466 |                  42495.6  |                         111 |                          0.0126 |                         4.1321 |
|         94 |                  2946.41   |                  20248.3  |                          83 |                          0.285  |                         1.9413 |
|         95 |                 -7377.36   |                  13806.4  |                          85 |                         -0.6986 |                         1.3171 |
|         96 |                 32705.1    |                  20557.8  |                          10 |                          3.385  |                         2.1379 |
|         97 |                  2626.91   |                  30820.1  |                         128 |                          0.3225 |                         3.1019 |
|         98 |                   662.375  |                  25630.1  |                         193 |                          0.1182 |                         2.5858 |
|         99 |                 15582.2    |                  39752.4  |                          95 |                          1.7101 |                         4.303  |
|        100 |                  6064.81   |                  26565.6  |                          49 |                          0.6272 |                         2.6221 |
|        101 |                   632.544  |                  17067.1  |                          94 |                          0.0674 |                         1.6703 |
|        102 |                  7490.89   |                  16957.8  |                         120 |                          0.7387 |                         1.6455 |
|        103 |                 -2622.3    |                  17689.1  |                          87 |                         -0.2433 |                         1.7054 |
|        104 |                  -524.018  |                  21346.3  |                          78 |                         -0.0561 |                         2.0823 |
|        105 |                 -2773.15   |                  13725.7  |                          76 |                         -0.2609 |                         1.314  |
|        106 |                  8148.66   |                  19641.4  |                         179 |                          0.8062 |                         1.9327 |
|        107 |                 -7552.97   |                  13625.6  |                          87 |                         -0.7267 |                         1.3193 |
|        108 |                  8012.81   |                  24891    |                         124 |                          0.7976 |                         2.4256 |
|        109 |                 -9100      |                    nan    |                           1 |                         -0.8922 |                       nan      |
|        110 |                 -7156.46   |                   7659.39 |                          16 |                         -0.6646 |                         0.7139 |
|        111 |                -12312.2    |                  15054    |                          19 |                         -1.1863 |                         1.4353 |
|        112 |                 -6387.15   |                  17772.4  |                         192 |                         -0.6047 |                         1.7086 |
|        113 |                 -6630.22   |                  13117.8  |                          82 |                         -0.6407 |                         1.2768 |
|        114 |                -24000      |                    nan    |                           1 |                         -2.3438 |                       nan      |
|        115 |                  8975.46   |                  23660.5  |                           6 |                          0.8868 |                         2.3202 |
|        116 |                 -6193.78   |                  31964    |                          16 |                         -0.5716 |                         3.0341 |
|        117 |                  2069.99   |                  13377.3  |                          92 |                          0.1982 |                         1.2778 |
|        118 |                -11078.1    |                  13807.8  |                         103 |                         -1.0565 |                         1.3197 |
|        119 |                 -5043.59   |                  23983.4  |                         192 |                         -0.485  |                         2.2957 |
|        120 |                  5313.72   |                  22209.8  |                         127 |                          0.5363 |                         2.1614 |
|        121 |                  8722.75   |                  13336.6  |                         107 |                          0.8561 |                         1.298  |
|        122 |                   684.604  |                  13169    |                          98 |                          0.0685 |                         1.2735 |
|        123 |                 -5750.99   |                  17840.8  |                         140 |                         -0.5429 |                         1.7341 |
|        124 |                  1718.56   |                  17465.4  |                          95 |                          0.1734 |                         1.6941 |
|        125 |                  6995.47   |                  22519.7  |                          81 |                          0.6976 |                         2.2359 |
|        126 |                  2974.15   |                  51335.7  |                          63 |                          0.3598 |                         5.0748 |
|        127 |                 32996.5    |                  66421.5  |                          15 |                          3.6716 |                         7.1959 |
|        128 |                  6902.17   |                  34937.7  |                         146 |                          0.7153 |                         3.5214 |
|        129 |                 12636.1    |                  15833.1  |                         208 |                          1.2497 |                         1.5782 |
|        130 |                 -5617.01   |                  14131.2  |                         158 |                         -0.5389 |                         1.3726 |
|        131 |                 -3705.96   |                   9652.2  |                          23 |                         -0.3361 |                         0.8926 |
|        132 |                 -4918.12   |                  24184    |                          53 |                         -0.4715 |                         2.3349 |

## 4. Observations Clés

- **Variabilité des poids**: CV = 6.38% (Bonne reproductibilité)
- **Biais Labo vs Auto**: 0.10% (Excellent accord)
- **Corrélation Labo-Auto**: 0.4903 (À améliorer)



GRAPHIQUES INTERACTIFS

